# VCM-only: architecture, training and Pi pipeline

The independent character-transcription model has been removed from this branch.
Only the existing v12 family encoder and finite heads are used by this notebook.
Unresolved parameters and custom media return clarification, not transcription.

The separate `phone_s53` research model uses constrained phoneme CTC for numbers
and dynamic catalogs; it is not automatically substituted for this v12 graph.
No acoustic weights were changed by removal. Historical validation is in Git
history. Run current tests before interpreting any prior accuracy as current.

Commands: `python train.py`, `python train.py --fit`, `python publish.py`.
Restored local VCM weights and referenced audio remain prerequisites; they are
not downloaded automatically. See `research/no_asr_review/RESULTS.md`.


## Current validation status

Transcription removal changes the pipeline. Historical run results are retained in Git history, not reported as a new run. Run `train.py` with the local VCM artifacts and human recordings to refresh.


## Architecture and contracts

```mermaid
flowchart LR
    A[Microphone / WAV] --> B[Energy VAD and endpoint]
    B --> C[16 kHz causal log-mel frontend]
    C --> D[VCM family encoder + finite parameter heads]
    D --> E[Family and per-value acceptance]
    E --> F[Typed command or clarification]
```

The v12 network remains unchanged: 398,422 parameters including its frozen
309,130-parameter family encoder. There is no independent transcription session,
transcription trainer, character decoder, or transcription payload. The source
classifier and the sequence-preserving research CNN are separate checkpoints.
Neither service executes device actions. Pi and live-microphone qualification
remain outstanding.


## Data and reproducibility

Training and validation read `data/vcm.jsonl` only. The original declared split
has 18,434 train, 5,364 development, 4,850 calibration and 4,314 test examples.
Those counts are historical metadata, not a newly replayed dataset.
The data audit checks hashes, speaker/pair/recording separation and only the audio
referenced by this VCM manifest. Unused files in an old shared audio index do not
become extra training requirements. Labels remain programmatic silver.

Finite-head continuation retains the encoder and uses train/development only;
calibration/test never select training weights. Parameter uncertainty cannot
invoke a second network. The original research packet must be restored separately
for the streaming phoneme experiments and visualization.


## Executable notebook source

Each `%%module` cell is ordinary Python source. `train.py` loads these modules in memory; `publish.py` extracts runtime cells into the executable archive. There is one implementation of each model and runtime component. The cell registry does not train models or open a microphone.


In [ ]:
from pathlib import Path
import sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Architecture.ipynb").is_file())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import train
SOURCES = train.register_sources(PROJECT)
def module_cell(line, cell):
    # Source was registered from this notebook; compile each displayed cell too.
    compile(cell, line.strip(), "exec")
get_ipython().register_magic_function(module_cell, "cell", "module")
print(f"Registered {len(SOURCES)} notebook modules")


## Runtime, endpointing and validation implementation


### `tinyvcm.__init__`


In [ ]:
%%module tinyvcm.__init__
__version__ = "1.0.0-experimental"


### `tinyvcm.audio`


In [ ]:
%%module tinyvcm.audio
"""One explicit frontend for training, test replay, and live inference."""
from __future__ import annotations

from dataclasses import dataclass, asdict
from functools import lru_cache
from math import gcd
from pathlib import Path
from tempfile import NamedTemporaryFile
import numpy as np
from einops import rearrange, reduce
from scipy.signal import resample_poly
import soundfile as sf


@dataclass(frozen=True)
class FrontendConfig:
    sample_rate: int = 16000
    window: int = 400
    hop: int = 160
    fft: int = 512
    mel_bins: int = 40
    fmin: float = 60.0
    fmax: float = 7600.0
    log_floor: float = 1e-8
    convention: str = "htk_peak_triangles_power_natural_log"


DEFAULT = FrontendConfig()


@lru_cache(maxsize=8)
def mel_filters(config=DEFAULT):
    frequencies = np.linspace(0, config.sample_rate / 2, config.fft // 2 + 1)
    low, high = 2595 * np.log10(1 + np.array([config.fmin, config.fmax]) / 700)
    boundaries = 700 * (10 ** (np.linspace(low, high, config.mel_bins + 2) / 2595) - 1)
    filters = np.zeros((config.mel_bins, len(frequencies)), dtype=np.float32)
    for m in range(config.mel_bins):
        a, b, c = boundaries[m:m + 3]
        filters[m] = np.maximum(0, np.minimum((frequencies - a) / (b - a), (c - frequencies) / (c - b)))
    return filters


def load_audio(path, channel=0):
    pcm, rate = sf.read(path, dtype="float32", always_2d=True)
    if not 0 <= channel < pcm.shape[1]:
        raise ValueError(f"Channel {channel} unavailable in {pcm.shape[1]}-channel audio")
    return resample_audio(pcm[:, channel], rate), rate


def resample_audio(pcm, rate, target=16000):
    pcm = np.asarray(pcm, dtype=np.float32)
    if pcm.ndim != 1:
        raise ValueError("Choose a microphone channel before resampling")
    if rate != target:
        common = gcd(int(rate), target)
        pcm = resample_poly(pcm, target // common, int(rate) // common).astype(np.float32)
    return pcm


def features(pcm, config=DEFAULT):
    """No future/center padding and no utterance-dependent normalization."""
    pcm = np.asarray(pcm, dtype=np.float32)
    if len(pcm) < config.window:
        pcm = np.pad(pcm, (0, config.window - len(pcm)))
    windows = np.lib.stride_tricks.sliding_window_view(pcm, config.window)[::config.hop]
    window = np.hanning(config.window + 1)[:-1].astype(np.float32)
    spectrum = np.fft.rfft(windows * window, n=config.fft, axis=-1)
    power = np.abs(spectrum).astype(np.float32) ** 2
    mel = np.einsum("tf,mf->mt", power, mel_filters(config), optimize=True)
    return np.log(np.maximum(mel, config.log_floor)).astype(np.float32)


class StreamingFrontend:
    def __init__(self, config=DEFAULT):
        self.config = config
        self.pending = np.empty(0, dtype=np.float32)

    def push(self, pcm):
        joined = np.empty(len(self.pending) + len(pcm), dtype=np.float32)
        joined[:len(self.pending)] = self.pending
        joined[len(self.pending):] = pcm
        if len(joined) < self.config.window:
            self.pending = joined
            return np.empty((self.config.mel_bins, 0), dtype=np.float32)
        frames = features(joined, self.config)
        self.pending = joined[frames.shape[1] * self.config.hop:].copy()
        return frames


def cache_path(record, root):
    import hashlib
    import json
    recipe = json.dumps(record.get("augmentation"), sort_keys=True) if record.get("augmentation") else ""
    key = hashlib.sha256((record["id"] + str(asdict(DEFAULT)) + recipe).encode()).hexdigest()
    return Path(root) / ".cache/features" / key[:2] / (key + ".npy")


def cached_features(record, root):
    target = cache_path(record, root)
    if target.exists():
        return np.load(target)
    pcm, _ = load_audio(Path(root) / record["audio"])
    array = features(pcm)
    target.parent.mkdir(parents=True, exist_ok=True)
    # Separate writers may prepare the same deterministic feature entry concurrently.
    # Publish a complete file atomically; never share a writable temporary filename.
    with NamedTemporaryFile(dir=target.parent, prefix=target.stem + "-", suffix=".tmp", delete=False) as f:
        temporary = Path(f.name)
        np.save(f, array, allow_pickle=False)
    try:
        try:
            temporary.rename(target)
        except FileExistsError:
            pass  # Another complete writer won the publication race on Windows.
    finally:
        temporary.unlink(missing_ok=True)
    return array


### `tinyvcm.levels`


In [ ]:
%%module tinyvcm.levels
"""Bounded local grammar for brightness/volume slots; no intent selection."""
from __future__ import annotations

import re

ONES = dict(zip("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen".split(), range(20)))
TENS = dict(zip("twenty thirty forty fifty sixty seventy eighty ninety".split(), range(20, 100, 10)))
NUMBER_WORDS = set(ONES) | set(TENS) | {"hundred", "thousand", "million"}
UP = r"\b(up|increase|raise|higher|more|brighten|brighter|louder|boost)\b"
DOWN = r"\b(down|decrease|reduce|lower|less|dim|dimmer|dimming|darker|darken|quieter|softer)\b"


def spoken_number(value: int) -> str:
    """Canonical English form, also used to enumerate the grammar in tests."""
    if type(value) is not int or not 0 <= value <= 100:
        raise ValueError("Percentage outside 0..100")
    if value == 100:
        return "one hundred"
    if value < 20:
        return next(word for word, n in ONES.items() if n == value)
    tens = next(word for word, n in TENS.items() if n == value // 10 * 10)
    return tens + (" " + spoken_number(value % 10) if value % 10 else "")


def _integer(words):
    if len(words) == 1 and words[0].isdigit():
        return int(words[0])
    if len(words) == 1 and words[0] in ONES:
        return ONES[words[0]]
    if len(words) == 1 and words[0] in TENS:
        return TENS[words[0]]
    if words in (["one", "hundred"], ["a", "hundred"], ["hundred"]):
        return 100
    if len(words) == 2 and words[0] in TENS and words[1] in ONES and 1 <= ONES[words[1]] <= 9:
        return TENS[words[0]] + ONES[words[1]]
    raise ValueError("Malformed or ambiguous numeric parameter")


def percentage(text: str) -> int | None:
    """Extract exactly one integer, preserving signs/decimals for rejection.

    None means no number was spoken. Invalid numbers must never become a
    default ten-point adjustment. Device names must be removed by the caller.
    """
    text = text.lower().replace("â€™", "'")
    text = re.sub(r"(?<=[a-z])-(?=[a-z])", " ", text)
    if re.search(r"\d[a-z]|[a-z]\d|\b(degrees?|decibels?|celsius|fahrenheit|watts?|seconds?|minutes?|hours?)\b", text):
        raise ValueError("Expected a percentage, not another unit or an ordinal")
    if re.search(r"\d\s*[.,/]\s*\d|[+\-âˆ’]\s*\d|\b(minus|negative|plus|point|decimal|half|quarter)\b", text):
        raise ValueError("Expected an unsigned whole percentage from 0 to 100")
    tokens = re.findall(r"\d+|[a-z]+|%", text)
    groups, current = [], []
    for i, token in enumerate(tokens):
        is_number = token.isdigit() or token in NUMBER_WORDS
        is_article = token == "a" and i + 1 < len(tokens) and tokens[i + 1] == "hundred"
        is_and = token == "and" and current and i + 1 < len(tokens) and tokens[i + 1] in NUMBER_WORDS
        if is_number or is_article or is_and:
            current.append(token)
        elif current:
            groups.append(current)
            current = []
    if current:
        groups.append(current)
    if not groups:
        if re.search(r"\b(percent|percentage|per cent)\b|%", text):
            raise ValueError("Missing numeric parameter before percent")
        return None
    if len(groups) != 1:
        raise ValueError("Ambiguous multiple numbers")
    value = _integer(groups[0])
    if not 0 <= value <= 100:
        raise ValueError("Percentage outside 0..100")
    return value


def level_parameter(text: str) -> dict:
    """Resolve a level-family slot to an absolute value or relative delta.

    Bare UP/DOWN means +/-10 percentage points. A numeric relative amount
    requires 'by'; 'to fifty' always means an absolute level, including in
    'turn the volume down to fifty'. An optional percent suffix is equivalent.
    """
    text = text.lower().replace("â€™", "'")
    if re.search(r"\b(not|don't|never|cancel|instead|unless|if|when|then|or)\b", text):
        raise ValueError("Contradictory, conditional, or ambiguous level parameter")
    if re.search(r"\band\b.*\b(play|turn|switch|set|change|dim|brighten)\b", text):
        raise ValueError("Multiple actions in one level request")
    if re.search(r"\b(what|which|whether|show|settings|status|current|double|halve)\b|\bhow (loud|bright|much)\b", text):
        raise ValueError("Unsupported level query or operation")
    if re.search(r"\b(after|before|until)\b|\b(in|at)\b.*\b(seconds?|minutes?|hours?|am|pm)\b", text):
        raise ValueError("Scheduled level changes are unsupported")
    if re.search(r"\b(volume|sound)\b", text) and re.search(r"\b(brightness|lights?|lamps?)\b", text):
        raise ValueError("Multiple device properties in one request")
    text = re.sub(r"\bdown (here|there)\b", "here", text)
    text = re.sub(r"\bless brighter\b", "less bright", text)
    up, down = bool(re.search(UP, text)), bool(re.search(DOWN, text))
    if up and down:
        raise ValueError("Conflicting level directions")
    value = percentage(text)
    if value is not None:
        by = bool(re.search(r"\bby\b", text))
        absolute = bool(re.search(r"\b(to|at|level)\s+(?:(?:a|the)\s+)?(?:level\s+)?(?:\d+|" + "|".join(sorted(NUMBER_WORDS)) + r")\b", text))
        if by:
            if absolute or not (up or down) or value == 0:
                raise ValueError("Relative amount requires one direction and a positive value")
            return {"delta_percentage_points": value if up else -value}
        if (up or down) and not absolute:
            raise ValueError("Say 'to' for an absolute level or 'by' for a relative amount")
        return {"percent": value}
    if re.search(r"\b(by|to|percent|percentage|max|maximum|minimum|full|medium|all the way)\b", text):
        raise ValueError("Missing or unsupported numeric level")
    if up or down:
        return {"delta_percentage_points": 10 if up else -10}
    raise ValueError("Missing level: say UP, DOWN, or a whole number from 0 to 100")


### `tinyvcm.routing`


In [ ]:
%%module tinyvcm.routing
"""Command selection belongs to the VCM; this module only validates arguments."""
from __future__ import annotations

import re
import unicodedata

from .levels import level_parameter, percentage

PARAMETER_COMMANDS = {"POWER_ON", "POWER_OFF", "BRIGHTNESS_SET", "BRIGHTNESS_UP", "BRIGHTNESS_DOWN", "COLOR_SET", "VOLUME_SET", "MEDIA_PLAY", "TIMER_SET"}
ACTION = {"POWER_ON": "SET_POWER", "POWER_OFF": "SET_POWER", "BRIGHTNESS_SET": "SET_BRIGHTNESS", "BRIGHTNESS_UP": "ADJUST_BRIGHTNESS", "BRIGHTNESS_DOWN": "ADJUST_BRIGHTNESS", "COLOR_SET": "SET_COLOR", "VOLUME_SET": "SET_VOLUME", "MEDIA_PLAY": "PLAY", "MEDIA_PAUSE": "PAUSE", "MEDIA_RESUME": "RESUME", "MEDIA_STOP": "STOP_MEDIA", "TIME_QUERY": "GET_TIME", "DATE_QUERY": "GET_DATE", "TIMER_SET": "CREATE_TIMER"}
PARAMETER_COMMANDS.update({"BRIGHTNESS", "VOLUME"})
ACTION.update({"BRIGHTNESS": "SET_BRIGHTNESS", "VOLUME": "SET_VOLUME"})
COLORS = {"red", "orange", "yellow", "green", "blue", "purple", "pink", "cyan", "white"}
ONES = dict(zip("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen".split(), range(20)))
TENS = dict(zip("twenty thirty forty fifty sixty seventy eighty ninety".split(), range(20, 100, 10)))


def number(text):
    tokens = re.findall(r"\d+|[a-z]+", text.lower())
    value = 0
    found = False
    for word in tokens:
        if word.isdigit():
            if found:
                raise ValueError("Ambiguous multiple numbers")
            value, found = int(word), True
        elif word in ONES:
            value, found = value + ONES[word], True
        elif word in TENS:
            value, found = value + TENS[word], True
        elif word == "hundred" and found:
            value *= 100
        elif word == "thousand" and found:
            value *= 1000
        elif word == "and":
            continue
        elif found:
            break
    if not found:
        raise ValueError("Missing numeric parameter")
    return value


def match_text(text):
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(c for c in text if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def without_target(text, target, *, preserve_numbers=False):
    remainder = match_text(text) if not preserve_numbers else "".join(
        c for c in unicodedata.normalize("NFKD", text.lower()) if not unicodedata.combining(c))
    for alias in sorted([target["name"], *target.get("aliases", [])], key=len, reverse=True):
        pattern = r"[^\w]+".join(re.escape(part) for part in match_text(alias).split())
        remainder = re.sub(r"\b" + pattern + r"\b", " ", remainder)
    return remainder


def match_target(text, catalog):
    normalized = " " + match_text(text) + " "
    matches = []
    for entity in catalog.get("entities", []):
        aliases = [entity["name"], *entity.get("aliases", [])]
        spans = [m.span() for alias in aliases for m in re.finditer(r"\b" + re.escape(match_text(alias)) + r"\b", normalized)]
        if spans:
            matches.append((entity, spans))
    if not matches:
        raise ValueError("Unknown or missing target name")
    # Exact full alias matching, with explicit ambiguity rather than nearest-name substitution.
    # A full alias owns an overlapping generic suffix ("bedroom lights" vs "lights").
    matches = [(entry, spans) for entry, spans in matches if not all(
        any(other["id"] != entry["id"] and a <= start and end <= b and (a, b) != (start, end)
            for other, other_spans in matches for a, b in other_spans)
        for start, end in spans)]
    if any(not entry.get("generic") for entry, _ in matches) and not re.search(r"\b(and|plus)\b", normalized):
        matches = [(entry, spans) for entry, spans in matches if not entry.get("generic")]
    unique = {entry["id"]: entry for entry, _ in matches}
    if len(unique) != 1:
        raise ValueError("Ambiguous target name")
    target = next(iter(unique.values()))
    if target.get("acoustic_key") and target["kind"] == "light" and re.search(r"\b(radio|fan|heater|thermostat|television|tv|computer)\b|\bair conditioner\b", normalized):
        raise ValueError("Spoken device conflicts with the registered light target")
    return target


def color_parameter(text, target=None):
    """Resolve one color independently of whether the complete command is valid."""
    remainder = without_target(text, target) if target else match_text(text)
    colors = COLORS & set(re.findall(r"[a-z]+", remainder))
    transitions = re.findall(r"\bfrom\s+(" + "|".join(sorted(COLORS)) + r")\s+to\s+(" + "|".join(sorted(COLORS)) + r")\b", remainder)
    if len(transitions) == 1 and colors == set(transitions[0]) and not re.search(r"\b(or|and)\b", remainder):
        return transitions[0][1]
    if len(colors) != 1:
        raise ValueError("Missing or ambiguous color parameter")
    return next(iter(colors))


def media_parameters(text):
    """Remove explicit request wording under a VCM-selected MEDIA_PLAY intent."""
    cleaned = text.lower().strip()
    cleaned = re.sub(r"\s+please$", "", cleaned)
    cleaned = re.sub(r"^please\s+", "", cleaned)
    if re.fullmatch(r"(?:resume|continue)(?:\s+(?:the\s+)?music)?", cleaned):
        raise ValueError("Parameter speech contradicts the VCM media intent")
    # Only strip these wrappers when followed by an explicit play request; names
    # such as 'I Want to Break Free' remain intact after an initial 'play'.
    cleaned = re.sub(r"^(?:(?:can|could|would|will) you (?:please )?|i (?:want|would like|would love) to )(?=(?:play|listen to|put on|start)\b)", "", cleaned)
    cleaned = re.sub(r"^(play|listen to|put on|start)\s+(?:for me\s+)?", "", cleaned)
    if re.fullmatch(r"(?:turn|switch)\s+(?:on\s+(?:the\s+)?music|(?:the\s+)?music\s+on)", cleaned):
        cleaned = "music"
    if not cleaned or cleaned in {"play", "listen to", "put on", "start"}:
        raise ValueError("Missing media parameter or explicit generic music request")
    if cleaned in {"music", "some music", "the music"}:
        return {}
    if " by " in cleaned:
        title, artist = cleaned.rsplit(" by ", 1)
        # An artist-only request does not name a song literally called 'songs'.
        if title.strip() not in {"music", "the music", "songs", "all songs", "some songs", "an album", "albums"}:
            return {"title": title.strip(), "artist": artist.strip()}
    return {"selection_text": cleaned}


def parse_parameters(command, text, catalog=None):
    if command not in ACTION:
        return {"status": "rejected", "reason": "unsupported_command"}
    catalog = catalog or {"entities": []}
    arguments = {}
    action = ACTION[command]
    try:
        if re.search(r"\b(do not|don't|never|instead|cancel)\b", text.lower()):
            raise ValueError("Contradictory or cancelled parameter tail")
        if command in {"POWER_ON", "POWER_OFF", "BRIGHTNESS", "BRIGHTNESS_SET", "BRIGHTNESS_UP", "BRIGHTNESS_DOWN", "COLOR_SET"}:
            target = match_target(text, catalog)
            arguments["target"] = {"id": target["id"], "name": target["name"], "device_kind": target["kind"]}
            if command.startswith("POWER_"):
                contradiction = "off" if command == "POWER_ON" else "on"
                if re.search(r"\b" + contradiction + r"\b", without_target(text, target)):
                    raise ValueError("Parameter tail contradicts the VCM power intent")
                arguments["on"] = command == "POWER_ON"
            elif target["kind"] != "light":
                raise ValueError("Requested light property on a non-light target")
        if command in {"BRIGHTNESS", "VOLUME"}:
            opposite = r"\b(volume|sound|louder|quieter)\b" if command == "BRIGHTNESS" else r"\b(brightness|lights?|lamps?|brighten|dim)\b"
            if re.search(opposite, text.lower()):
                raise ValueError("Parameter speech contradicts the VCM device property")
            numeric_text = without_target(text, target, preserve_numbers=True) if command == "BRIGHTNESS" else text
            if command == "BRIGHTNESS" and COLORS & set(re.findall(r"[a-z]+", numeric_text)):
                raise ValueError("Color speech contradicts the VCM brightness family")
            parameter = level_parameter(numeric_text)
            arguments.update(parameter)
            action = ("SET_" if "percent" in parameter else "ADJUST_") + command
        elif command in {"BRIGHTNESS_SET", "VOLUME_SET"}:
            numeric_text = text.lower()
            if command == "BRIGHTNESS_SET":
                numeric_text = without_target(text, target, preserve_numbers=True)
            value = percentage(numeric_text)
            if value is None:
                raise ValueError("Missing numeric parameter")
            arguments["percent"] = value
        elif command in {"BRIGHTNESS_UP", "BRIGHTNESS_DOWN"}:
            numeric_text = without_target(text, target, preserve_numbers=True)
            number_words = "|".join([*ONES, *TENS])
            if re.search(r"\b(to|at|on)\s+(?:(?:a|the)\s+)?(?:level\s+)?(?:\d+|" + number_words + r"|max(?:imum)?|full|medium|half)\b|\bhalf way\b|\ball the way\b", numeric_text):
                raise ValueError("Absolute brightness cannot be executed as a relative adjustment")
            direction_text = re.sub(r"\bdown (here|there)\b", "here", numeric_text)
            direction_text = re.sub(r"\bless brighter\b", "less bright", direction_text)
            opposite = r"\b(dim|dimmer|decrease|reduce|lower|down|less|quieter|softer)\b" if command.endswith("_UP") else r"\b(brighten|brighter|increase|raise|up|higher|louder)\b"
            if re.search(opposite, direction_text):
                raise ValueError("Parameter speech contradicts the VCM brightness direction")
            value = percentage(numeric_text)
            value = 10 if value is None else value
            if not 0 < value <= 100:
                raise ValueError("Adjustment outside 1..100 percentage points")
            arguments["delta_percentage_points"] = value * (1 if command.endswith("_UP") else -1)
        elif command == "COLOR_SET":
            arguments["color"] = color_parameter(text, target)
        elif command == "TIMER_SET":
            value = number(text)
            units = [unit for unit in ("hour", "minute", "second") if unit in text.lower()]
            if len(units) != 1:
                raise ValueError("Specify one duration unit")
            value *= {"hour": 3600, "minute": 60, "second": 1}[units[0]]
            if not 0 < value <= 86400:
                raise ValueError("Duration outside 1..86400 seconds")
            arguments["duration_s"] = value
        elif command == "MEDIA_PLAY":
            arguments = media_parameters(text)
        return validate_arguments(command, arguments, catalog, action=action)
    except ValueError as exc:
        return {"status": "needs_clarification", "vcm_command": command, "reason": str(exc)}




def validate_arguments(command, arguments, catalog=None, *, action=None):
    """Shared typed checks for typed acoustic-slot arguments."""
    catalog = catalog or {"entities": []}
    if command not in ACTION:
        return {"status": "rejected", "reason": "unsupported_command"}
    try:
        args = dict(arguments)
        allowed = {"POWER_ON": {"target", "on"}, "POWER_OFF": {"target", "on"},
                   "BRIGHTNESS_UP": {"target", "delta_percentage_points"}, "BRIGHTNESS_DOWN": {"target", "delta_percentage_points"},
                   "BRIGHTNESS": {"target", "percent", "delta_percentage_points"}, "VOLUME": {"percent", "delta_percentage_points"},
                   "BRIGHTNESS_SET": {"target", "percent"}, "VOLUME_SET": {"percent"}, "COLOR_SET": {"target", "color"},
                   "MEDIA_PLAY": {"selection_text", "title", "artist"}, "TIMER_SET": {"duration_s"}}
        if set(args) - allowed.get(command, set()):
            raise ValueError("Unexpected parameter for the VCM intent")
        if command in {"POWER_ON", "POWER_OFF", "BRIGHTNESS", "BRIGHTNESS_SET", "BRIGHTNESS_UP", "BRIGHTNESS_DOWN", "COLOR_SET"}:
            target = args.get("target", {})
            if not isinstance(target, dict):
                raise ValueError("Invalid target binding")
            matches = [e for e in catalog.get("entities", []) if e["id"] == target.get("id")]
            if len(matches) != 1:
                raise ValueError("Unknown or ambiguous target binding")
            entity = matches[0]
            args["target"] = {"id": entity["id"], "name": entity["name"], "device_kind": entity["kind"]}
            if not command.startswith("POWER_") and entity["kind"] != "light":
                raise ValueError("Requested light property on a non-light target")
        if command.startswith("POWER_") and args.get("on") is not (command == "POWER_ON"):
            raise ValueError("Power parameter contradicts the VCM intent")
        if command == "COLOR_SET" and args.get("color") not in COLORS:
            raise ValueError("Missing or unsupported color")
        if command in {"BRIGHTNESS", "VOLUME", "BRIGHTNESS_SET", "VOLUME_SET"}:
            fields = {"percent", "delta_percentage_points"} & set(args)
            if len(fields) != 1 or command.endswith("_SET") and "percent" not in fields:
                raise ValueError("Specify exactly one supported level operation")
        if command == "MEDIA_PLAY" and (any(not isinstance(v, str) or not v.strip() for v in args.values())
                or "selection_text" in args and len(args) != 1):
            raise ValueError("Invalid media selection")
        if command in {"BRIGHTNESS_UP", "BRIGHTNESS_DOWN"}:
            value = args.get("delta_percentage_points")
            if type(value) is not int or not 0 < abs(value) <= 100 or (value > 0) != command.endswith("_UP"):
                raise ValueError("Relative parameter contradicts the VCM intent")
        if "percent" in args and (type(args["percent"]) is not int or not 0 <= args["percent"] <= 100):
            raise ValueError("Percentage outside 0..100")
        if "delta_percentage_points" in args and (type(args["delta_percentage_points"]) is not int or not 0 < abs(args["delta_percentage_points"]) <= 100):
            raise ValueError("Adjustment outside 1..100 percentage points")
        return {"status": "accepted", "command": {"action": action or ACTION[command], "arguments": args}}
    except ValueError as exc:
        return {"status": "needs_clarification", "vcm_command": command, "reason": str(exc)}


def parameter_slice(pcm, trigger_seconds, lookback_seconds=0.5, rate=16000):
    start = max(0, int(round((trigger_seconds - lookback_seconds) * rate)))
    if start >= len(pcm):
        return pcm[:0], start
    return pcm[start:], start


### `tinyvcm.slot_contract`


In [ ]:
%%module tinyvcm.slot_contract
"""Finite acoustic slots and typed validation. Never selects or changes an intent."""
from __future__ import annotations

import hashlib
import json
from .routing import COLORS, color_parameter, match_target, parse_parameters, validate_arguments

LABELS = ["UNKNOWN", "POWER_ON", "POWER_OFF", "BRIGHTNESS_UP", "BRIGHTNESS_DOWN",
          "COLOR_SET", "MEDIA_PLAY", "MEDIA_RESUME", "TIME_QUERY", "DATE_QUERY"]
TARGET_COMMANDS = {"POWER_ON", "POWER_OFF", "BRIGHTNESS_UP", "BRIGHTNESS_DOWN", "COLOR_SET"}
RELATIVE_COMMANDS = {"BRIGHTNESS_UP", "BRIGHTNESS_DOWN"}
PARAMETERLESS = {"MEDIA_RESUME", "TIME_QUERY", "DATE_QUERY"}


def catalog_signature(catalog):
    """IDs may be rebound; changing the spoken vocabulary invalidates acoustic slots."""
    vocabulary = sorted((e.get("acoustic_key", e["id"]), e["name"].lower(), e["kind"],
                         sorted(a.lower() for a in e.get("aliases", [])), bool(e.get("generic")))
                        for e in catalog.get("entities", []))
    return hashlib.sha256(json.dumps(vocabulary, sort_keys=True).encode()).hexdigest()


def vocabulary(catalog):
    return {
        "guard": ["READY", "MISSING", "UNRESOLVED", "CONFLICT"],
        "target": ["NONE", "OTHER", *[e.get("acoustic_key", e["id"]) for e in catalog["entities"]]],
        "color": ["NONE", "OTHER", *sorted(COLORS)],
        "amount": ["NONE", "OTHER", *[str(n) for n in range(1, 101)]],
        "media": ["NONE", "OTHER", "GENERIC"],
    }


def supervision(command, text, catalog):
    """Training/evaluation only: conservative silver labels from human transcripts."""
    slots = {"guard": "CONFLICT", "target": "NONE", "color": "NONE", "amount": "NONE", "media": "NONE"}
    parsed = parse_parameters(command, text, catalog)
    if command == "UNKNOWN":
        return slots, parsed
    # Parameter evidence is independent: a missing target must not teach the
    # color head that an audible color is absent, or vice versa. The complete
    # parser result still governs readiness and the executable reference.
    target = None
    if command in TARGET_COMMANDS:
        try:
            target = match_target(text, catalog)
            slots["target"] = target.get("acoustic_key", target["id"])
        except ValueError:
            slots["target"] = "OTHER"
    if command == "COLOR_SET":
        try:
            slots["color"] = color_parameter(text, target)
        except ValueError:
            slots["color"] = "OTHER"
    if command in RELATIVE_COMMANDS:
        from .routing import without_target
        from .levels import percentage
        try:
            # Unknown target names may contain numbers; do not infer absence.
            value = percentage(without_target(text, target, preserve_numbers=True)) if target else None
            slots["amount"] = "OTHER" if target is None else "NONE" if value is None else str(value)
        except ValueError:
            slots["amount"] = "OTHER"
    if parsed["status"] != "accepted":
        reason = parsed.get("reason", "")
        if "Unknown or missing target" in reason:
            slots["guard"] = "UNRESOLVED"
        elif "Missing or ambiguous color" in reason:
            slots.update(guard="UNRESOLVED" if not COLORS.intersection(text.lower().split()) else "CONFLICT", color="OTHER")
        elif "Missing media" in reason:
            slots.update(guard="MISSING", media="NONE")
        return slots, parsed
    arguments = parsed["command"]["arguments"]
    slots["guard"] = "READY"
    if command in TARGET_COMMANDS:
        target = next(e for e in catalog["entities"] if e["id"] == arguments["target"]["id"])
        slots["target"] = target.get("acoustic_key", target["id"])
    if command == "COLOR_SET":
        slots["color"] = arguments["color"]
    if command in RELATIVE_COMMANDS:
        # Explicit ten and an omitted amount must remain different labels.
        from .routing import without_target
        from .levels import percentage
        value = percentage(without_target(text, target, preserve_numbers=True))
        slots["amount"] = "NONE" if value is None else str(value)
    if command == "MEDIA_PLAY":
        slots["media"] = "OTHER" if arguments else "GENERIC"
        if arguments:
            slots["guard"] = "UNRESOLVED"
    return slots, parsed


def assemble(command, values, catalog):
    """Convert confidently decoded slots to validated typed arguments, without text."""
    args = {}
    if command in TARGET_COMMANDS:
        matches = [e for e in catalog.get("entities", []) if e.get("acoustic_key", e["id"]) == values.get("target")]
        if len(matches) != 1:
            return {"status": "needs_clarification", "vcm_command": command, "reason": "Unresolved acoustic target"}
        target = matches[0]
        args["target"] = {"id": target["id"], "name": target["name"], "device_kind": target["kind"]}
    if command.startswith("POWER_"):
        args["on"] = command == "POWER_ON"
    if command in RELATIVE_COMMANDS:
        value = values.get("amount")
        if value == "NONE":
            value = 10
        elif value not in {str(n) for n in range(1, 101)}:
            return {"status": "needs_clarification", "vcm_command": command, "reason": "Unresolved spoken amount"}
        args["delta_percentage_points"] = int(value) * (1 if command.endswith("_UP") else -1)
    if command == "COLOR_SET":
        args["color"] = values.get("color")
    if command == "MEDIA_PLAY" and values.get("media") != "GENERIC":
        return {"status": "needs_clarification", "vcm_command": command, "reason": "Unresolved media selection"}
    return validate_arguments(command, args, catalog)


def required_heads(command):
    heads = ["guard"]
    if command in TARGET_COMMANDS:
        heads.append("target")
    if command in RELATIVE_COMMANDS:
        heads.append("amount")
    if command == "COLOR_SET":
        heads.append("color")
    if command == "MEDIA_PLAY":
        heads.append("media")
    return heads


def acoustic_candidate(command, probabilities, vocab, catalog):
    values, confidence = {}, {}
    for name in required_heads(command):
        p = probabilities[name]
        index = max(range(len(p)), key=p.__getitem__)
        values[name], confidence[name] = vocab[name][index], float(p[index])
    result = assemble(command, values, catalog) if values["guard"] == "READY" else None
    score = min(confidence.values()) if result and result["status"] == "accepted" else 0.
    return {"values": values, "confidence": confidence, "score": score, "result": result}


def apply_value_policy(candidate, policy):
    """An unvalidated vocabulary entry cannot inherit a common value's threshold."""
    thresholds = policy.get("value_thresholds")
    if thresholds is None:
        return candidate
    unsupported = {name: value for name, value in candidate["values"].items() if name != "guard"
                   and candidate["confidence"][name] < thresholds.get(name, {}).get(value, 1.1)}
    return {**candidate, "score": 0. if unsupported else candidate["score"], "unsupported_values": unsupported}


def route_acoustic(command, probabilities, config, catalog, variant="fp32", *, intent_confidence=1.):
    if config.get("schema_version") == "family-finite-v1":
        from .family_contract import route_family
        return route_family(command, probabilities, config, catalog, variant)
    policy = config.get("variants", {}).get(variant, {})
    if catalog_signature(catalog) != config["catalog_signature"]:
        return {"decision": "fallback", "reason": "acoustic_catalog_changed"}
    candidate = apply_value_policy(acoustic_candidate(command, probabilities, config["vocabulary"], catalog), policy)
    if command == "MEDIA_PLAY" and config.get("custom_media") and candidate["values"].get("media") == "GENERIC":
        mode = probabilities.get("media_mode", [])
        names = config["vocabulary"].get("media_mode", [])
        if len(mode) != len(names) or not len(mode):
            candidate = {**candidate, "score": 0.}
        else:
            index = max(range(len(mode)), key=mode.__getitem__)
            if names[index] != "GENERIC" or mode[index] < policy.get("generic_media_threshold", 1.1):
                candidate = {**candidate, "score": 0.}
    if candidate["score"] >= policy.get("acceptance_thresholds", {}).get(command, 1.1) and intent_confidence >= policy.get("joint_intent_thresholds", {}).get(command, 0.):
        return {"decision": "accepted", "candidate": candidate, **candidate["result"]}
    guard = candidate["values"]["guard"]
    if guard in {"MISSING", "CONFLICT"} and candidate["confidence"]["guard"] >= policy.get("clarification_thresholds", {}).get(guard, 1.1):
        return {"decision": "clarify", "status": "needs_clarification", "vcm_command": command,
                "reason": "acoustic_parameter_" + guard.lower(), "candidate": candidate}
    return {"decision": "fallback", "reason": "unresolved_acoustic_parameters", "candidate": candidate}




def agrees_with_acoustic(command, candidate, arguments, config, catalog, variant):
    if not candidate:
        return True
    policy = config.get("variants", {}).get(variant, {})
    thresholds = policy.get("field_thresholds", {})
    for name, value in candidate["values"].items():
        if name == "guard" or value in {"NONE", "OTHER"} or candidate["confidence"][name] < thresholds.get(name, 1.1):
            continue
        if "value_thresholds" in policy and candidate["confidence"][name] < policy["value_thresholds"].get(name, {}).get(value, 1.1):
            continue
        if name == "target":
            matches = [e for e in catalog.get("entities", []) if e.get("acoustic_key", e["id"]) == value]
            if len(matches) != 1 or arguments.get("target", {}).get("id") != matches[0]["id"]:
                return False
        elif name == "color" and arguments.get("color") != value:
            return False
        elif name == "amount" and abs(arguments.get("delta_percentage_points", 0)) != int(value):
            return False
        elif name == "media" and value == "GENERIC" and arguments:
            return False
    return True


### `tinyvcm.family_contract`


In [ ]:
%%module tinyvcm.family_contract
"""Versioned family/finite-head research contract; transcripts are labels only."""
from __future__ import annotations

import re
from typing import Any

from .levels import level_parameter, percentage
from .routing import match_target, parse_parameters, without_target, validate_arguments
from .slot_contract import supervision, vocabulary, assemble, catalog_signature, apply_value_policy

LABELS = ["UNKNOWN", "POWER_ON", "POWER_OFF", "BRIGHTNESS", "COLOR_SET",
          "MEDIA_PLAY", "MEDIA_RESUME", "TIME_QUERY", "DATE_QUERY", "VOLUME"]


def contract(catalog: dict[str, Any]) -> dict[str, Any]:
    heads = vocabulary(catalog)
    heads["operation"] = ["INCREASE", "DECREASE", "SET", "MUTE", "MIN", "MAX"]
    heads["amount"] = ["NONE", "OTHER", *map(str, range(101))]
    heads["media_mode"] = ["GENERIC", "CUSTOM"]
    return {"schema_version": "family-finite-v1", "intent_labels": LABELS,
            "vocabulary": heads, "streaming_supported": False,
            "annotation_states": {"null": "unannotated; zero loss", "NONE": "explicitly absent",
                                  "OTHER": "unsupported or unresolved value"},
            "defaults": {"relative_step": {"value": 10, "unit": "percentage_points",
                         "requires": ["accepted INCREASE/DECREASE", "accepted amount NONE"]}},
            "operations": {"INCREASE": "positive relative delta_percentage_points",
                           "DECREASE": "negative relative delta_percentage_points", "SET": "absolute percent",
                           "MUTE": "distinct operation; execution disabled",
                           "MIN": "distinct operation; execution disabled", "MAX": "distinct operation; execution disabled"},
            "custom_media": {"schema_version": 1, "open_text_fields": ["title", "artist"],
                             "required_finite_heads": []},
            "unsupported": ["unmute", "arbitrary numbers", "unrestricted device names", "parameter-only audio timing"],
            "activation": "Every finite head disabled until calibrated support/precision and joint-command checks pass",
            "color_gate": {"minimum_accepts": 20, "observed_precision": .99},
            "reference_quality": "programmatic silver; no human adjudication"}


def annotate(row: dict[str, Any], catalog: dict[str, Any]) -> dict[str, Any]:
    """Retain independent known fields and distinguish unknown from absence."""
    family = row["command"]
    if family not in LABELS:
        raise ValueError(f"Unsupported family: {family}")
    text = row.get("transcript")
    labels = {name: None for name in contract(catalog)["vocabulary"]}
    if text is None:
        return {"labels": labels, "reference": None, "quality": "unannotated"}
    reference = parse_parameters(family, text, catalog)
    old, _ = supervision(family, text, catalog)
    labels["guard"] = old["guard"]
    if family in {"POWER_ON", "POWER_OFF", "COLOR_SET"}:
        labels["target"] = old["target"]
    if family == "COLOR_SET":
        labels["color"] = old["color"]
    if family == "MEDIA_PLAY":
        labels["media"] = old["media"]
        if reference["status"] == "accepted":
            labels["media_mode"] = "CUSTOM" if reference["command"]["arguments"] else "GENERIC"
    if family in {"BRIGHTNESS", "VOLUME"}:
        remainder = text
        if family == "BRIGHTNESS":
            try:
                target = match_target(text, catalog)
                labels["target"] = target.get("acoustic_key", target["id"])
                remainder = without_target(text, target, preserve_numbers=True)
            except ValueError:
                labels["target"] = "OTHER"
        try:
            level = level_parameter(remainder)
            if "percent" in level:
                labels["operation"], labels["amount"] = "SET", str(level["percent"])
            else:
                labels["operation"] = "INCREASE" if level["delta_percentage_points"] > 0 else "DECREASE"
                explicit = percentage(remainder)
                labels["amount"] = "NONE" if explicit is None else str(explicit)
        except ValueError:
            # No guessed direction or numeric default. These endpoint labels are
            # research supervision only and cannot emit an executable command.
            if re.fullmatch(r"(?:volume )?mute(?: (?:the )?(?:volume|sound))?", remainder.strip().lower()):
                labels["operation"] = "MUTE"
            elif re.search(r"\b(max|maximum|full)\b", remainder.lower()):
                labels["operation"] = "MAX"
            elif re.search(r"\b(minimum)\b", remainder.lower()):
                labels["operation"] = "MIN"
            labels["amount"] = "OTHER"
        labels["guard"] = "READY" if reference["status"] == "accepted" else "UNRESOLVED"
    return {"labels": labels, "reference": reference, "quality": "programmatic_silver"}


def legacy_action(family: str, operation: str) -> str:
    """Only exact operation mappings are legal; absolute is never relative."""
    mapping = {("BRIGHTNESS", "INCREASE"): "BRIGHTNESS_UP",
               ("BRIGHTNESS", "DECREASE"): "BRIGHTNESS_DOWN",
               ("BRIGHTNESS", "SET"): "BRIGHTNESS_SET", ("VOLUME", "SET"): "VOLUME_SET"}
    if (family, operation) not in mapping:
        raise ValueError("No exact legacy action mapping")
    return mapping[family, operation]


def required_heads(family: str) -> list[str]:
    heads = ["guard"]
    if family in {"POWER_ON", "POWER_OFF", "BRIGHTNESS", "COLOR_SET"}:
        heads.append("target")
    if family in {"BRIGHTNESS", "VOLUME"}:
        heads += ["operation", "amount"]
    if family == "COLOR_SET":
        heads.append("color")
    if family == "MEDIA_PLAY":
        heads += ["media", "media_mode"]
    return heads


def assemble_family(family: str, values: dict, catalog: dict) -> dict:
    if family not in {"BRIGHTNESS", "VOLUME"}:
        if family == "MEDIA_PLAY" and values.get("media_mode") != "GENERIC":
            return {"status": "needs_parameters", "reason": "custom_media_name_required"}
        return assemble(family, values, catalog)
    operation, amount = values.get("operation"), values.get("amount")
    if operation not in {"INCREASE", "DECREASE", "SET"}:
        return {"status": "needs_parameters", "reason": "unsupported_or_missing_operation"}
    args = {}
    if family == "BRIGHTNESS":
        matches = [e for e in catalog.get("entities", []) if e.get("acoustic_key", e["id"]) == values.get("target")]
        if len(matches) != 1:
            return {"status": "needs_parameters", "reason": "unresolved_target"}
        args["target"] = {"id": matches[0]["id"]}
    if amount == "NONE" and operation in {"INCREASE", "DECREASE"}:
        amount = "10"
    if amount not in {str(n) for n in range(101)}:
        return {"status": "needs_parameters", "reason": "unresolved_amount"}
    if operation == "SET":
        args["percent"] = int(amount)
    else:
        args["delta_percentage_points"] = int(amount) * (1 if operation == "INCREASE" else -1)
    action = ("SET_" if operation == "SET" else "ADJUST_") + family
    return validate_arguments(family, args, catalog, action=action)


def family_candidate(family: str, probabilities: dict, config: dict, catalog: dict) -> dict:
    values, confidence = {}, {}
    for name in required_heads(family):
        p = probabilities[name]
        index = max(range(len(p)), key=p.__getitem__)
        values[name] = config["vocabulary"][name][index]
        confidence[name] = float(p[index])
    result = assemble_family(family, values, catalog) if values["guard"] == "READY" else None
    score = min(confidence.values()) if result and result["status"] == "accepted" else 0.
    return {"values": values, "confidence": confidence, "score": score, "result": result}


def route_family(family: str, probabilities: dict, config: dict, catalog: dict, variant: str) -> dict:
    if catalog_signature(catalog) != config.get("catalog_signature"):
        return {"decision": "fallback", "reason": "acoustic_catalog_changed"}
    policy = config.get("variants", {}).get(variant, {})
    candidate = apply_value_policy(family_candidate(family, probabilities, config, catalog), policy)
    if candidate["score"] >= policy.get("acceptance_thresholds", {}).get(family, 1.1):
        return {"decision": "accepted", "candidate": candidate, **candidate["result"]}
    guard = candidate["values"]["guard"]
    if guard in {"MISSING", "CONFLICT"} and candidate["confidence"]["guard"] >= policy.get("clarification_thresholds", {}).get(guard, 1.1):
        return {"decision": "clarify", "status": "needs_clarification",
                "reason": "acoustic_parameter_"+guard.lower(), "candidate": candidate}
    return {"decision": "fallback", "reason": "unresolved_acoustic_parameters", "candidate": candidate}


### `tinyvcm.bundle`


In [ ]:
%%module tinyvcm.bundle
"""Shared bundle contracts for runtime, service preflight and integrity checks."""
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

from .routing import ACTION


def runtime_components(metadata: dict[str, Any]) -> list[str]:
    """Honor explicit capabilities; retain the historical bundle convention."""
    vcm = metadata.get("vcm", {})
    streaming = vcm.get("streaming_supported", not bool(metadata.get("acoustic_slots")))
    if type(streaming) is not bool:
        raise ValueError("streaming_supported must be a boolean")
    if metadata.get("acoustic_slots") and streaming:
        raise ValueError("Acoustic slot bundles do not support streaming")
    components = metadata.get("runtime_components")
    if components is None:
        return ["vcm", *( ["vcm_stream"] if streaming else [])]
    if (not isinstance(components, list)
            or not all(isinstance(item, str) for item in components)
            or "vcm" not in components or len(set(components)) != len(components)
            or set(components) - {"vcm", "vcm_stream"}):
        raise ValueError("Invalid runtime_components; VCM is mandatory")
    if ("vcm_stream" in components) != streaming:
        raise ValueError("Streaming capability must match runtime_components")
    return components


def validate_labels(metadata: dict[str, Any]) -> list[str]:
    labels = metadata.get("vcm", {}).get("labels")
    if (not isinstance(labels, list) or not labels
            or not all(isinstance(label, str) for label in labels)
            or len(set(labels)) != len(labels) or "UNKNOWN" not in labels
            or set(labels) - {"UNKNOWN", *ACTION}):
        raise ValueError("Invalid explicit VCM label vocabulary")
    slots = metadata.get("acoustic_slots")
    if slots:
        from .slot_contract import LABELS
        expected = slots.get("intent_labels", LABELS)
        if labels != expected:
            raise ValueError("Acoustic slot intent labels/order do not match VCM labels")
        vocab = slots.get("vocabulary", {})
        if not vocab or any(not values or len(values) != len(set(values))
                            for values in vocab.values()):
            raise ValueError("Invalid acoustic head vocabulary")
    for policy in [metadata.get("routing", {}),
                   *metadata.get("routing", {}).get("variants", {}).values()]:
        if set(policy.get("acceptance_thresholds", {})) - set(labels):
            raise ValueError("Acceptance policy contains labels outside the VCM vocabulary")
    return labels


def load_runtime_contract(bundle: str | Path, variant: str = "fp32") -> dict[str, Any]:
    """Validate the selected VCM path without a transcription component."""
    if variant not in {"fp32", "int8"}:
        raise ValueError("variant must be fp32 or int8")
    bundle = Path(bundle)
    path = bundle / "bundle.json"
    if not path.is_file():
        raise FileNotFoundError(f"Missing model file: {path}; select a bundle with --bundle")
    metadata = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(metadata, dict) or not isinstance(metadata.get("vcm"), dict):
        raise ValueError("Bundle metadata must contain a VCM object")
    if type(metadata.get("schema_version", 1)) is not int or metadata.get("schema_version", 1) != 1:
        raise ValueError("Unsupported bundle schema_version")
    components = runtime_components(metadata)
    validate_labels(metadata)
    for component in components:
        path = bundle / f"{component}.{variant}.onnx"
        if not path.is_file():
            raise FileNotFoundError(f"Missing model file: {path}; select a bundle with --bundle")
    return metadata


def validate_outputs(session: Any, metadata: dict[str, Any]) -> None:
    """Check named graph heads against the declared vocabularies, never by size alone."""
    outputs = {output.name: output.shape for output in session.get_outputs()}
    expected = {"logits": len(metadata["vcm"]["labels"])}
    expected.update({f"slot_{name}": len(values) for name, values in
                     metadata.get("acoustic_slots", {}).get("vocabulary", {}).items()})
    for name, size in expected.items():
        if name not in outputs or len(outputs[name]) != 2 or outputs[name][-1] != size:
            raise ValueError(f"ONNX output {name} does not match its declared vocabulary")
    # New exports can bind exact label order into ONNX; legacy exports lack this.
    declared = session.get_modelmeta().custom_metadata_map.get("vcm_labels")
    if declared is not None and json.loads(declared) != metadata["vcm"]["labels"]:
        raise ValueError("ONNX label order does not match bundle metadata")
    slot_vocab = session.get_modelmeta().custom_metadata_map.get("slot_vocabulary")
    if slot_vocab is not None and json.loads(slot_vocab) != metadata.get("acoustic_slots", {}).get("vocabulary"):
        raise ValueError("ONNX acoustic head vocabulary does not match bundle metadata")


### `tinyvcm.inference`


In [ ]:
%%module tinyvcm.inference
"""Offline CPU inference and a causal intent-to-parameter handoff."""
from __future__ import annotations

import json
from pathlib import Path
import time

import numpy as np
from einops import rearrange
import onnxruntime as ort

from .audio import features, load_audio
from .bundle import load_runtime_contract, runtime_components, validate_outputs
from .routing import PARAMETER_COMMANDS, parse_parameters


def softmax(values, temperature=1.0):
    values = np.asarray(values, dtype=np.float64) / temperature
    exponents = np.exp(values - values.max())
    return exponents / exponents.sum()


class Recognizer:
    def __init__(self, bundle, threads=2, variant="fp32"):
        self.bundle = Path(bundle)
        self.metadata = load_runtime_contract(self.bundle, variant)
        self.labels = self.metadata["vcm"]["labels"]
        options = ort.SessionOptions()
        options.intra_op_num_threads = threads
        options.inter_op_num_threads = 1
        self.options = options
        self.variant = variant
        self.vcm = ort.InferenceSession(str(self.bundle / f"vcm.{variant}.onnx"), sess_options=options, providers=["CPUExecutionProvider"])
        validate_outputs(self.vcm, self.metadata)
        self.stream = ort.InferenceSession(str(self.bundle / f"vcm_stream.{variant}.onnx"), sess_options=options, providers=["CPUExecutionProvider"]) if "vcm_stream" in runtime_components(self.metadata) else None
        self.catalog = self.metadata.get("catalog", {"entities": []})

    def routing_policy(self):
        policy = self.metadata.get("routing", {})
        return {**policy, **policy.get("variants", {}).get(self.variant, {})}

    def normalize(self, mel, task="vcm"):
        stats = self.metadata[task]["normalization"]
        mean = rearrange(np.array(stats["mean"], dtype=np.float32), "m -> m 1")
        std = rearrange(np.array(stats["std"], dtype=np.float32), "m -> m 1")
        return rearrange((mel - mean) / std, "m t -> 1 m t")

    def classify(self, pcm, *, temperature=None):
        if not len(pcm):
            raise ValueError("An utterance must contain audio samples")
        mel = self.normalize(features(pcm))
        logits = self.vcm.run(["logits"], {"features": mel, "lengths": np.array([mel.shape[-1]], dtype=np.int64)})[0]
        return softmax(logits[0], self.routing_policy().get("temperature", 1.0) if temperature is None else temperature)

    def classify_with_slots(self, pcm, *, temperature=None):
        if not len(pcm):
            raise ValueError("An utterance must contain audio samples")
        mel = features(pcm)
        x = self.normalize(mel)
        names = list(self.metadata["acoustic_slots"]["vocabulary"])
        outputs = self.vcm.run(["logits", *["slot_" + name for name in names]], {"features": x, "lengths": np.array([x.shape[-1]], dtype=np.int64)})
        slots = {name: softmax(value[0]) for name, value in zip(names, outputs[1:])}
        return softmax(outputs[0][0], self.routing_policy().get("temperature", 1.0) if temperature is None else temperature), slots, mel

    def stream_events(self, pcm):
        state = self.begin_stream()
        self.advance_stream(pcm, state, final=True)
        return state["events"]

    def begin_stream(self):
        shapes = [] if self.metadata.get("acoustic_slots") or not self.metadata["vcm"].get("streaming_supported", True) else self.metadata["vcm"]["state_shapes"]
        fills = self.metadata["vcm"].get("state_initial_values", [0.0] * len(shapes))
        return {"states": [np.full(shape, fill, dtype=np.float32) for shape, fill in zip(shapes, fills)], "frame": 0, "previous": None, "stable": 0, "events": []}

    def advance_stream(self, pcm, state, final=False):
        """Process only new frames; live capture retains 80 ms for resampling edges."""
        policy = self.routing_policy()
        if self.metadata.get("acoustic_slots") or not self.metadata.get("vcm", {}).get("streaming_supported", True) or policy.get("trigger_threshold", .9) > 1:
            return []  # A disabled provisional trigger cannot produce an event.
        mel = self.normalize(features(pcm))
        states = state["states"]
        threshold = policy.get("trigger_threshold", 0.9)
        required = policy.get("stability_chunks", 3)
        minimum = policy.get("minimum_audio_seconds", 0.5)
        chunk = 4
        previous, stable = state["previous"], state["stable"]
        events = state["events"]
        before = len(events)
        available = mel.shape[-1] if final else max(0, (mel.shape[-1] - 8) // chunk * chunk)
        for frame in range(state["frame"], available, chunk):
            x = mel[:, :, frame:min(frame + chunk, available)]
            if x.shape[-1] == 0:
                break
            feed = {"features": x, **{f"state_{i}": state for i, state in enumerate(states)}}
            logits, *states = self.stream.run(None, feed)
            probabilities = softmax(logits[0], policy.get("temperature", 1.0))
            index = int(probabilities.argmax())
            command, confidence = self.labels[index], float(probabilities[index])
            timestamp = (frame + x.shape[-1] - 1) * 0.01 + 0.025
            stable = (stable + 1 if command == previous else 1) if confidence >= threshold else 0
            previous = command
            if not events and command != "UNKNOWN" and confidence >= threshold and stable >= required and timestamp >= minimum:
                events.append({"stage": "intent", "vcm_command": command, "confidence": confidence, "audio_time_s": timestamp, "provisional": True})
            state["frame"] = frame + x.shape[-1]
        state.update(states=states, previous=previous, stable=stable)
        return events[before:]



    def predict(self, pcm, parameter_start_ms=None, stream=True, events=None):
        started = time.perf_counter()
        pcm = np.asarray(pcm, dtype=np.float32)
        if pcm.ndim != 1 or not np.isfinite(pcm).all():
            return {"status": "rejected", "reason": "invalid_audio"}
        if not 0.1 <= len(pcm) / 16000 <= 8.2:
            return {"status": "rejected", "reason": "audio_duration_outside_supported_range"}
        slot_config = self.metadata.get("acoustic_slots")
        slot_probabilities, shared_mel, slot_decision = None, None, None
        if slot_config and slot_config.get("enabled", True):
            probabilities, slot_probabilities, shared_mel = self.classify_with_slots(pcm)
        else:
            probabilities = self.classify(pcm)
        index = int(probabilities.argmax())
        command, confidence = self.labels[index], float(probabilities[index])
        policy = self.routing_policy()
        threshold = policy.get("acceptance_thresholds", {}).get(command, policy.get("acceptance_threshold", 0.7))
        if slot_probabilities is not None and command != "UNKNOWN" and confidence >= threshold:
            from .slot_contract import route_acoustic
            slot_decision = route_acoustic(command, slot_probabilities, slot_config, self.catalog, self.variant, intent_confidence=confidence)
        if command == "UNKNOWN" or confidence < threshold:
            return {"status": "rejected", "reason": "unknown_or_low_confidence", "vcm_command": command, "confidence": confidence, "intent_accepted": False, "processing_ms": (time.perf_counter()-started)*1000}
        events = events if events is not None else self.stream_events(pcm) if stream else []
        event = events[0] if events else None
        result = {"vcm_command": command, "confidence": confidence, "events": events,
                  "intent_accepted": True, "intent_acceptance_source": "intent"}
        if event and event["vcm_command"] != command:
            result.update(status="rejected", reason="early_intent_retracted", intent_accepted=False)
        elif slot_decision is not None and slot_decision["decision"] == "clarify":
            result.update(status="needs_clarification", reason=slot_decision["reason"], parameter_source="vcm")
        elif command not in PARAMETER_COMMANDS:
            result.update(parse_parameters(command, "", self.catalog))
            result["parameter_source"] = "none"
        else:
            if parameter_start_ms is not None and not 0 <= parameter_start_ms / 1000 < len(pcm) / 16000:
                raise ValueError("Parameter boundary is outside the recording")
            if slot_probabilities is not None:
                result["acoustic_parameters"] = {k: v for k, v in slot_decision.items() if k in {"decision", "reason", "candidate"}}
                if slot_decision["decision"] in {"accepted", "clarify"}:
                    result.update({k: v for k, v in slot_decision.items() if k in {"status", "command", "reason"}})
                    result.update(parameter_source="vcm", processing_ms=(time.perf_counter()-started)*1000)
                    return result
            result.update(status="needs_parameters", parameter_source="vcm",
                          reason="catalog_decoder_required" if command == "MEDIA_PLAY"
                          else "finite_parameters_require_clarification")
        result["processing_ms"] = (time.perf_counter()-started)*1000
        return result


### `tinyvcm.text`


In [ ]:
%%module tinyvcm.text
"""Small runtime text helpers; importing these never loads PyTorch."""



import re
import unicodedata

def normalize_text(text):
    text = text.lower().replace("’", "'")
    text = "".join(c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", re.sub(r"[^a-z' ]", " ", text)).strip()


### `tinyvcm.config`


In [ ]:
%%module tinyvcm.config
import ast
import math
from pathlib import Path
import tomllib

def resolve_switches(values):
    """Interpret only boolean names, and/or/not and parentheses. Never eval()."""
    resolved, visiting = {}, set()
    def switch(name):
        if name in resolved:
            return resolved[name]
        if name not in values:
            raise ValueError(f"Unknown boolean switch: {name}")
        if name in visiting:
            raise ValueError(f"Cyclic boolean expression: {name}")
        visiting.add(name)
        value = values[name]
        if type(value) is bool:
            result = value
        elif isinstance(value, str):
            def walk(node):
                if isinstance(node, ast.Name):
                    if node.id.lower() in {"true", "false"}:
                        return node.id.lower() == "true"
                    return switch(node.id)
                if isinstance(node, ast.Constant) and type(node.value) is bool:
                    return node.value
                if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.Not):
                    return not walk(node.operand)
                if isinstance(node, ast.BoolOp) and isinstance(node.op, (ast.And, ast.Or)):
                    results = [walk(item) for item in node.values]
                    return all(results) if isinstance(node.op, ast.And) else any(results)
                raise ValueError("Use only booleans, switch names, and/or/not and parentheses")
            result = walk(ast.parse(value, mode="eval").body)
        else:
            raise ValueError(f"Switch {name} must be a boolean or boolean expression")
        visiting.remove(name)
        resolved[name] = result
        return result
    for name in values:
        switch(name)
    return resolved

def load_config(path):
    config = tomllib.loads(Path(path).read_text(encoding="utf-8"))
    expected = {"workflow", "tests", "runtime", "validation", "training", "publish", "audio"}
    if set(config) != expected:
        raise ValueError(f"Configuration sections must be {sorted(expected)}")
    for section, keys in {"workflow": {"setup", "train", "test", "validate", "report"},
                          "tests": {"vad", "vcm", "pipeline"},
                          "runtime": {"vad", "vcm"}}.items():
        if set(config[section]) != keys:
            raise ValueError(f"Unknown or missing {section} switches")
        config[section] = resolve_switches(config[section])
    if any(k in config["training"] for k in ("asr", "asr_epochs")):
        raise ValueError("Transcription training switches have been removed")
    for key in ("vcm", "update_artifacts"):
        if type(config["training"][key]) is not bool:
            raise ValueError(f"training.{key} must be true or false")
    for section, key in (("validation", "require_quality_gate"), ("publish", "include_wheels")):
        if type(config[section][key]) is not bool:
            raise ValueError(f"{section}.{key} must be true or false")
    if config["validation"]["split"] not in {"dev", "calibration", "test"}:
        raise ValueError("Validation split must be dev, calibration or test")
    if type(config["validation"]["threads"]) is not int or not 1 <= config["validation"]["threads"] <= 4:
        raise ValueError("Use one to four inference threads")
    for section, keys in {"validation": ("max_examples",), "training": ("max_train_examples", "max_dev_examples")}.items():
        if any(type(config[section][k]) is not int or config[section][k] < 0 for k in keys):
            raise ValueError("Example limits must be non-negative integers")
    if any(type(config["training"][k]) is not int or config["training"][k] < 1
           for k in ("vcm_epochs", "batch_size", "patience")):
        raise ValueError("Epochs, batch size and patience must be positive integers")
    if config["training"]["device"] not in {"cpu", "cuda"}:
        raise ValueError("training.device must be cpu or cuda")
    if type(config["training"]["seed"]) is not int or config["training"]["seed"] < 0:
        raise ValueError("training.seed must be a nonnegative integer")
    learning_rate = config["training"]["learning_rate"]
    if type(learning_rate) not in (int, float) or not math.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError("training.learning_rate must be a finite positive number")
    versions = config["publish"]["python_versions"]
    if not versions or len(set(versions)) != len(versions) or set(versions) - {"3.11", "3.12", "3.13"}:
        raise ValueError("Choose unique publish Python versions from 3.11, 3.12 and 3.13")
    if type(config["publish"]["sample_per_intent"]) is not int or config["publish"]["sample_per_intent"] < 1:
        raise ValueError("publish.sample_per_intent must be positive")
    audio = config["audio"]
    if audio["sample_rate"] != 16000 or audio["block_ms"] != 40:
        raise ValueError("The current checkpoint uses 16 kHz and 40 ms capture blocks")
    if any(not isinstance(audio[k], (int, float)) or type(audio[k]) is bool or not math.isfinite(audio[k]) or audio[k] <= 0
           for k in ("preroll_ms", "silence_ms", "max_utterance_seconds", "energy_floor", "noise_multiplier")):
        raise ValueError("Audio parameters must be positive numbers")
    if audio["max_utterance_seconds"] > 8 or audio["silence_ms"] >= audio["max_utterance_seconds"] * 1000:
        raise ValueError("Endpointing must stay inside the current eight-second model contract")
    return config


### `tinyvcm.metrics`


In [ ]:
%%module tinyvcm.metrics
from collections import Counter
import numpy as np

def wilson(good: int, count: int) -> list[float] | None:
    if not count:
        return None
    z = 1.959963984540054
    p = good / count
    center = (p + z*z/(2*count)) / (1 + z*z/count)
    radius = z*np.sqrt(p*(1-p)/count + z*z/(4*count*count)) / (1 + z*z/count)
    return [float(center-radius), float(center+radius)]

def metrics(records: list[dict], labels: list[str]) -> dict:
    counts = {c: Counter() for c in labels}
    confusion = {c: Counter({label: 0 for label in labels}) for c in labels}
    for row in records:
        gold, predicted, accepted = row["gold"], row["predicted"], row["accepted"]
        counts[gold]["support"] += 1
        confusion[gold][predicted] += 1
        counts[gold]["raw_correct"] += gold == predicted
        counts[gold]["abstentions"] += not accepted
        if accepted:
            counts[predicted]["accepted_predictions"] += 1
            counts[predicted]["wrong_accepts"] += gold != predicted
            counts[gold]["correct_accepted"] += gold == predicted
            counts[gold]["false_accepts_on_reference"] += gold != predicted
    good = sum(v["correct_accepted"] for v in counts.values())
    emitted = sum(v["accepted_predictions"] for v in counts.values())
    for c, row in counts.items():
        n, a = row["support"], row["accepted_predictions"]
        row["raw_recall"] = row["raw_correct"] / n if n else None
        row["correct_accepted_recall"] = row["correct_accepted"] / n if n else None
        row["accepted_precision"] = row["correct_accepted"] / a if a else None
        row["raw_recall_wilson95"] = wilson(row["raw_correct"], n)
        row["correct_accepted_recall_wilson95"] = wilson(row["correct_accepted"], n)
        row["accepted_precision_wilson95"] = wilson(row["correct_accepted"], a)
    unknown = counts["UNKNOWN"]
    low_raw = [c for c, r in counts.items() if r["raw_recall"] is None or r["raw_recall"] < .8]
    low_accepted = [c for c, r in counts.items() if c != "UNKNOWN" and
                    (r["correct_accepted_recall"] is None or r["correct_accepted_recall"] < .6)]
    low_precision = [c for c, r in counts.items() if c != "UNKNOWN" and
                     (r["accepted_precision"] is None or r["accepted_precision"] < .95)]
    unknown_rate = unknown["false_accepts_on_reference"] / max(1, unknown["support"])
    precision = good / max(1, emitted)
    return {"per_family": counts, "confusion": confusion,
            "correct_accepted": good, "accepted": emitted, "accepted_precision": precision,
            "unknown_false_accepts": unknown["false_accepts_on_reference"],
            "unknown_support": unknown["support"], "unknown_false_accept_rate": unknown_rate,
            "unknown_rate_wilson95": wilson(unknown["false_accepts_on_reference"], unknown["support"]),
            "gates": {"low_raw_recall": low_raw, "low_accepted_recall": low_accepted,
                      "additional_low_per_intent_precision": low_precision,
                      "global_precision_pass": precision >= .95, "unknown_pass": unknown_rate <= .01,
                      "inherited_pass": not low_raw and not low_accepted and precision >= .95 and unknown_rate <= .01},
            "uncertainty_note": "Wilson intervals describe recording counts, not independent-speaker or seed uncertainty."}

def policy_rows(records: list[dict], labels: list[str], policy: dict) -> list[dict]:
    from tinyvcm.inference import softmax
    output = []
    for row in records:
        p = softmax(np.log(np.maximum(row["probabilities"], 1e-30)), policy.get("temperature", 1.))
        command = labels[int(p.argmax())]
        confidence = float(p.max())
        output.append({**row, "predicted": command, "confidence": confidence,
                       "accepted": command != "UNKNOWN" and confidence >= policy["acceptance_thresholds"][command]})
    return output


### `tinyvcm.pipeline`


In [ ]:
%%module tinyvcm.pipeline
"""Shared endpointing for microphone and replay, with independently selectable stages."""
from collections import deque
import numpy as np
from einops import pack, reduce


class EnergyVAD:
    def __init__(self, config, rate=16000):
        self.config, self.rate = config, rate
        self.noise = .003
        self.reset()

    def reset(self):
        self.preroll = deque(maxlen=max(1, round(self.config["preroll_ms"] / self.config["block_ms"])))
        self.buffer, self.active, self.elapsed, self.silence = [], False, 0., 0.

    def push(self, chunk):
        """Return (PCM, timed_out) on an endpoint, otherwise None."""
        chunk = np.asarray(chunk, dtype=np.float32)
        if chunk.ndim != 1 or not len(chunk) or not np.isfinite(chunk).all():
            raise ValueError("VAD expects finite mono audio")
        duration = len(chunk) / self.rate
        rms = float(np.sqrt(reduce(chunk * chunk, "t ->", "mean")))
        speech = rms > max(self.config["energy_floor"], self.noise * self.config["noise_multiplier"])
        if not self.active:
            self.preroll.append(chunk.copy())
            if not speech:
                self.noise = .98 * self.noise + .02 * rms
                return None
            self.active, self.buffer = True, list(self.preroll)
            self.elapsed = sum(len(c) for c in self.buffer) / self.rate
        else:
            self.buffer.append(chunk.copy())
            self.elapsed += duration
        self.silence = 0. if speech else self.silence + duration
        timeout = self.elapsed + 1e-9 >= self.config["max_utterance_seconds"]
        if timeout or self.silence * 1000 + 1e-9 >= self.config["silence_ms"]:
            pcm, _ = pack(self.buffer, "*")
            self.reset()
            return pcm, timeout
        return None

    def segment(self, pcm):
        """Replay as a stream, including trailing silence to close the last utterance."""
        self.reset()
        self.noise = .003  # Independent recordings must not inherit another recording's noise estimate.
        size = round(self.rate * self.config["block_ms"] / 1000)
        padded = np.pad(pcm, (0, round(self.rate * (self.config["silence_ms"] + self.config["block_ms"]) / 1000)))
        output = []
        for offset in range(0, len(padded), size):
            result = self.push(padded[offset:offset + size])
            if result is not None:
                output.append(result)
        return output


class Pipeline:
    def __init__(self, bundle, config):
        self.config = config
        switches = config["runtime"]
        self.vad = EnergyVAD(config["audio"]) if switches["vad"] else None
        self.recognizer = None
        if switches["vcm"]:
            from .inference import Recognizer
            self.recognizer = Recognizer(bundle, threads=config["validation"]["threads"])

    def utterance(self, pcm, timeout=False):
        if timeout:
            return {"status": "rejected", "reason": "utterance_timeout"}
        if self.recognizer:
            result = self.recognizer.predict(pcm, stream=False)
            required = {"POWER_ON": ["target"], "POWER_OFF": ["target"], "BRIGHTNESS": ["target", "level"],
                        "COLOR_SET": ["target", "color"], "MEDIA_PLAY": ["selection_text"], "VOLUME": ["level"],
                        "MEDIA_RESUME": [], "TIME_QUERY": [], "DATE_QUERY": []}
            known = result.get("intent_accepted", False) and result.get("status") != "rejected"
            family = result.get("vcm_command") if known else None
            fields = required.get(family) if known else None
            return {**result, "stage": "result", "provisional": False, "intent": family,
                    "required_parameters": fields,
                    "parameters_needed": bool(fields) and result.get("status") != "accepted" if known else None}
        return {"status": "speech_detected", "seconds": len(pcm) / 16000}

    def process(self, pcm):
        if self.vad:
            segments = self.vad.segment(pcm)
            return [self.utterance(*item) for item in segments] or [
                {"status": "rejected", "reason": "no_speech"}]
        if not any(self.config["runtime"].values()):
            return [{"status": "disabled", "reason": "all_runtime_stages_disabled"}]
        return [self.utterance(pcm)]


### `tinyvcm.capture`


In [ ]:
%%module tinyvcm.capture
"""Numeric microphone selection and capture using the same VAD as replay."""
import json
import queue
import sys


def resolve_microphone(devices, hostapis, selection=None, host_api=None, default=None):
    selection = default if selection is None else selection
    if selection is None or not str(selection).isdigit():
        raise ValueError("Select a numeric microphone index from --list-mics")
    index = int(selection)
    if not 0 <= index < len(devices) or devices[index]["max_input_channels"] < 1:
        raise ValueError("Selected index is not an input device")
    return index, devices[index]


def listen(pipeline, args):
    import sounddevice as sd
    from .audio import resample_audio
    from .pipeline import EnergyVAD
    if not pipeline.config["runtime"]["vad"]:
        raise ValueError("Microphone capture requires runtime.vad = true; use --wav for presegmented audio")
    devices, hosts = sd.query_devices(), sd.query_hostapis()
    index, device = resolve_microphone(devices, hosts, args.mic, default=sd.default.device[0])
    if not 0 <= args.channel < device["max_input_channels"]:
        raise ValueError("Input channel unavailable")
    rate = args.capture_rate or 16000
    try:
        sd.check_input_settings(device=index, channels=args.channel + 1, samplerate=rate, dtype="float32")
    except sd.PortAudioError:
        if args.capture_rate:
            raise
        rate = int(device["default_samplerate"])
        sd.check_input_settings(device=index, channels=args.channel + 1, samplerate=rate, dtype="float32")
    print(json.dumps({"microphone": device["name"], "index": index, "capture_rate": rate,
                      "model_rate": 16000, "channel": args.channel}), file=sys.stderr, flush=True)
    chunks, overflow = queue.Queue(maxsize=200), [False]
    def callback(indata, frames, timing, status):
        if status:
            overflow[0] = True
        try:
            chunks.put_nowait(indata[:, args.channel].copy())
        except queue.Full:
            overflow[0] = True
    vad = EnergyVAD(pipeline.config["audio"], rate)
    blocksize = round(rate * pipeline.config["audio"]["block_ms"] / 1000)
    with sd.InputStream(device=index, samplerate=rate, channels=args.channel + 1,
                        dtype="float32", blocksize=blocksize, callback=callback) as stream:
        while stream.active:
            try:
                chunk = chunks.get(timeout=3)
            except queue.Empty as exc:
                raise RuntimeError("Microphone disconnected or stalled") from exc
            if overflow[0]:
                vad.reset()
                while not chunks.empty():
                    try:
                        chunks.get_nowait()
                    except queue.Empty:
                        break
                overflow[0] = False
                print(json.dumps({"status": "rejected", "reason": "audio_overflow"}), flush=True)
                continue
            was_active = vad.active
            result = vad.push(chunk)
            if not was_active and vad.active:
                print(json.dumps({"stage": "vad", "status": "speech_started"}), flush=True)
            if result is not None:
                pcm, timeout = result
                print(json.dumps(pipeline.utterance(resample_audio(pcm, rate), timeout),
                                 ensure_ascii=False, allow_nan=False), flush=True)


### `tinyvcm.validation`


In [ ]:
%%module tinyvcm.validation
"""Current component checks and held-out evaluation; no historical test tree."""
from collections import Counter, defaultdict
import copy
import hashlib
import json
from pathlib import Path
import time
import numpy as np
from .audio import load_audio
from .pipeline import EnergyVAD, Pipeline


def read_rows(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def edit_distance(a, b):
    previous = list(range(len(b) + 1))
    for i, left in enumerate(a, 1):
        current = [i]
        for j, right in enumerate(b, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (left != right)))
        previous = current
    return previous[-1]


def run_checks(config, bundle):
    """Real graph smoke checks plus explicitly labelled routing fixtures."""
    from .config import resolve_switches
    checks = {}
    def require(condition, message):
        if not condition:
            raise AssertionError(message)
    require(resolve_switches({"vad": True, "vcm": True, "pipeline": "vad and vcm"})["pipeline"],
            "Boolean expression resolution")
    for bad in ({"a": "__import__('os')"}, {"a": "b", "b": "a"}, {"a": "missing"}, {"a": 1}):
        try:
            resolve_switches(bad)
        except (ValueError, SyntaxError):
            pass
        else:
            raise AssertionError("Invalid boolean expression accepted")
    checks["config"] = {"status": "passed", "safe_boolean_expressions": True}
    pcm = (.1 * np.sin(2 * np.pi * 220 * np.arange(16000) / 16000)).astype(np.float32)
    if config["tests"]["vad"]:
        vad = EnergyVAD(config["audio"])
        require(not vad.segment(np.zeros(16000, dtype=np.float32)), "Silence triggered VAD")
        sequence = np.pad(pcm, (8000, 8000))
        from einops import pack
        twice, _ = pack([sequence, sequence], "*")
        found = vad.segment(twice)
        require(len(found) == 2 and not any(timeout for _, timeout in found), "Repeated VAD endpointing")
        from einops import repeat
        found = vad.segment(repeat(pcm, "t -> (r t)", r=10))
        require(found and found[0][1], "Overlong speech did not time out")
        from .capture import resolve_microphone
        devices = [{"max_input_channels": 0}, {"max_input_channels": 2}]
        require(resolve_microphone(devices, [], 1)[0] == 1, "Numeric microphone selection")
        try:
            resolve_microphone(devices, [], 0)
        except ValueError:
            pass
        else:
            raise AssertionError("Output-only device accepted")
        # Simulated PortAudio callbacks exercise the actual live-capture loop at
        # 48 kHz, including channel selection, fallback, overflow and two endpoints.
        import contextlib
        import io
        import sys
        from types import SimpleNamespace
        from unittest.mock import patch
        from .capture import listen
        rate = 48000
        tone = (.1 * np.sin(2 * np.pi * 220 * np.arange(rate) / rate)).astype(np.float32)
        native, _ = pack([np.pad(tone, (rate//2, rate//2)), np.pad(tone, (rate//2, rate//2))], "*")
        block = round(rate * config["audio"]["block_ms"] / 1000)
        class FakePortAudioError(Exception):
            pass
        class FakeStream:
            def __init__(self, **kwargs):
                self.kwargs, self.offset = kwargs, 0
            def __enter__(self):
                return self
            def __exit__(self, *exc):
                return False
            @property
            def active(self):
                if self.offset >= len(native):
                    return False
                audio = np.zeros((min(block, len(native)-self.offset), 2), dtype=np.float32)
                audio[:, 1] = native[self.offset:self.offset+block]
                self.kwargs["callback"](audio, len(audio), None, self.offset == 0)
                self.offset += len(audio)
                return True
        def input_settings(**kwargs):
            if kwargs["samplerate"] == 16000:
                raise FakePortAudioError("native rate only")
        fake = SimpleNamespace(InputStream=FakeStream, PortAudioError=FakePortAudioError,
            check_input_settings=input_settings, default=SimpleNamespace(device=(1, 1)),
            query_hostapis=lambda: [{"name": "Simulated"}],
            query_devices=lambda: [{"max_input_channels": 0}, {"max_input_channels": 2,
                "name": "Simulated input", "hostapi": 0, "default_samplerate": rate}])
        lengths = []
        cfg = copy.deepcopy(config); cfg["runtime"]["vad"] = True
        stub = SimpleNamespace(config=cfg, utterance=lambda audio, timeout:
            (lengths.append(len(audio)) or {"status": "fixture", "timeout": timeout}))
        stdout, stderr = io.StringIO(), io.StringIO()
        with patch.dict(sys.modules, {"sounddevice": fake}), contextlib.redirect_stdout(stdout), contextlib.redirect_stderr(stderr):
            listen(stub, SimpleNamespace(mic=1, channel=1, capture_rate=None))
        output = [json.loads(line) for line in stdout.getvalue().splitlines()]
        require(len(lengths) == 2 and all(16000 <= n < 32000 for n in lengths), "Native microphone resampling/endpoints")
        require(any(item.get("reason") == "audio_overflow" for item in output), "Overflow must be rejected")
        require(json.loads(stderr.getvalue())["capture_rate"] == 48000, "Native-rate fallback")
        checks["vad"] = {"status": "passed", "silence": True, "repeated_utterances": 2,
                         "timeout": True, "numeric_microphone": True, "simulated_native_capture_rate": 48000,
                         "simulated_capture_endpoints": len(lengths), "overflow_rejected": True, "live_microphone": False}
    if config["tests"]["vcm"]:
        from .inference import Recognizer
        model = Recognizer(bundle, threads=config["validation"]["threads"])
        scores = model.classify(pcm)
        require(np.isfinite(scores).all() and abs(float(scores.sum()) - 1) < 1e-6, "VCM probabilities")
        silence = model.predict(np.zeros(16000, dtype=np.float32), stream=False)
        require(silence["status"] == "rejected" , "Silence must abstain")
        require(model.stream is None, "VCM-only path loaded another model")
        checks["vcm"] = {"status": "passed", "labels": model.labels, "silence": silence["status"],
                         "providers": model.vcm.get_providers()}
    if config["tests"]["pipeline"]:
        cfg = copy.deepcopy(config)
        cfg["runtime"] = {"vad": True, "vcm": True}
        pipeline = Pipeline(bundle, cfg)
        require(pipeline.process(np.zeros(16000, dtype=np.float32))[0]["reason"] == "no_speech", "VAD bypass")
        results = pipeline.process(pcm)
        require(results and all("status" in r for r in results), "Full pipeline result")
        # Inject probabilities into an independent in-memory copy to check handoff.
        # Production artifacts and their disabled custom-media policy remain untouched.
        model = pipeline.recognizer
        original = copy.deepcopy(model.metadata)
        vocab = model.metadata["acoustic_slots"]["vocabulary"]
        def onehot(values, value):
            p = np.zeros(len(values)); p[values.index(value)] = 1.; return p
        slot = {name: onehot(values, values[0]) for name, values in vocab.items()}
        slot["guard"] = onehot(vocab["guard"], "READY")
        slot["media_mode"] = onehot(vocab["media_mode"], "CUSTOM")
        slot["media"] = onehot(vocab["media"], "OTHER")
        family = ["MEDIA_PLAY"]
        model.classify_with_slots = lambda audio: (onehot(model.labels, family[0]), slot, None)
        result = model.predict(pcm, stream=False)
        require(result["status"] == "needs_parameters", "Custom media must clarify")
        require(result["reason"] == "catalog_decoder_required", "Unexpected fallback")
        require(not hasattr(model, "transcribe"), "Transcription API reintroduced")
        family[0] = "UNKNOWN"
        require(model.predict(pcm, stream=False)["status"] == "rejected", "Unknown accepted")
        model.metadata = original
        checks["pipeline"] = {"status": "passed", "real_vad_vcm_replay": True,
            "routing_fixtures": ["custom media clarifies", "UNKNOWN rejected"],
            "production_thresholds_modified": False}
    return checks


def audit_data(root):
    root = Path(root)
    descriptor = json.loads((root / "data/dataset.json").read_text(encoding="utf-8"))
    index = json.loads((root / "data/audio-index.json").read_text(encoding="utf-8"))
    splits = {}
    required_audio = set()
    for task in ("vcm",):
        path = root / f"data/{task}.jsonl"
        if hashlib.sha256(path.read_bytes()).hexdigest() != descriptor["current_manifests"][task]:
            raise ValueError(f"Changed {task} manifest; update the reviewed dataset descriptor first")
        rows = read_rows(path)
        splits[task] = dict(Counter(r["split"] for r in rows))
        required_audio.update(row["audio"] for row in rows)
        for field in ("id", "audio", "speaker_id", "paired_group"):
            groups = defaultdict(set)
            for row in rows:
                if row.get(field):
                    groups[row[field]].add(row["split"])
            if any(len(s) > 1 for s in groups.values()):
                raise ValueError(f"{task}: {field} leaks across dataset splits")
        if any(row["audio"] not in index for row in rows):
            raise ValueError("Manifest audio missing from the integrity index")
    for relative in sorted(required_audio):
        expected = index[relative]
        path = (root / relative).resolve()
        if not path.is_relative_to((root / "data/audio").resolve()) or not path.is_file():
            raise ValueError(f"Missing or invalid current audio path: {relative}")
        with path.open("rb") as stream:
            actual = hashlib.file_digest(stream, "sha256").hexdigest()
        if actual != expected["sha256"] or path.stat().st_size != expected["bytes"]:
            raise ValueError(f"Changed audio: {relative}")
    if hashlib.sha256((root / "data/references.jsonl").read_bytes()).hexdigest() != descriptor["references_sha256"]:
        raise ValueError("Reference labels changed")
    return {"audio_files_verified": len(required_audio), "split_counts": splits,
            "recording_speaker_pair_split_leaks": 0, "reference_quality": "programmatic silver"}


def evaluate(config, bundle, rows, root, references):
    from .metrics import metrics
    from .text import normalize_text
    started = time.perf_counter()
    selected = config["tests"]
    vcm = pipeline = None
    if selected["vcm"]:
        from .inference import Recognizer
        vcm = Recognizer(bundle, threads=config["validation"]["threads"])
    if selected["pipeline"]:
        cfg = copy.deepcopy(config); cfg["runtime"] = {"vad": True, "vcm": True}
        pipeline = Pipeline(bundle, cfg)
    vad = EnergyVAD(config["audio"]) if selected["vad"] else None
    records, endpoint_records = [], []
    speech_count = segments_count = timeouts = 0
    command_counts = {name: Counter() for name in ("vcm", "pipeline")}
    predictions = []
    for i, row in enumerate(rows, 1):
        pcm, rate = load_audio(Path(root) / row["audio"])
        entry = {"id": row["id"], "gold": row["command"]}
        if vad:
            segments = vad.segment(pcm)
            speech_count += bool(segments); segments_count += len(segments)
            timeouts += sum(int(timeout) for _, timeout in segments)
        reference = references.get(row["id"], {}).get("reference")
        finite = reference and reference.get("status") == "accepted" and not (
            set(reference["command"]["arguments"]) & {"title", "artist", "selection_text"})
        for name, engine, output in (("vcm", vcm, records), ("pipeline", pipeline, endpoint_records)):
            if engine is None:
                continue
            results = [engine.predict(pcm, stream=False)] if name == "vcm" else engine.process(pcm)
            # All segments count as emissions; multiple endpoints cannot earn a complete-command success.
            result = results[0] if len(results) == 1 else {"status": "rejected", "reason": "multiple_endpoints"}
            output.append({"gold": row["command"], "predicted": result.get("vcm_command", "UNKNOWN"),
                           "accepted": result.get("intent_accepted", False)})
            counts = command_counts[name]
            counts["finite_references"] += bool(finite)
            counts["emitted"] += sum(r.get("status") == "accepted" and "command" in r for r in results)
            counts["correct_complete"] += bool(finite and result.get("status") == "accepted"
                and result.get("vcm_command") == row["command"] and result.get("command") == reference["command"])
            entry[name] = results
        predictions.append(entry)
        if i % 200 == 0:
            print(f"VALIDATE {i}/{len(rows)} elapsed={time.perf_counter()-started:.0f}s", flush=True)
    report = {"examples": len(rows), "elapsed_seconds": time.perf_counter()-started,
              "quality_gate_passed": False, "target_hardware_measured": False}
    for name, engine, output in (("vcm", vcm, records), ("pipeline", pipeline, endpoint_records)):
        if engine is not None:
            labels = (engine if name == "vcm" else engine.recognizer).labels
            counts = command_counts[name]
            report[name] = {"intent": metrics(output, labels), "commands": dict(counts),
                "exact_finite_command_recall": counts["correct_complete"] / max(1, counts["finite_references"]),
                "accepted_command_precision": counts["correct_complete"] / max(1, counts["emitted"])}
    if vad:
        report["vad"] = {"recordings_with_speech": speech_count, "endpoints": segments_count, "timeouts": timeouts,
                         "boundary_accuracy_assessed": False}
    command_paths = [report[name] for name in ("vcm", "pipeline") if name in report]
    report["quality_gate_passed"] = bool(command_paths) and all(
        part["intent"]["gates"]["inherited_pass"]
        and not part["intent"]["gates"]["additional_low_per_intent_precision"]
        and part["accepted_command_precision"] >= .95
        and part["exact_finite_command_recall"] >= .60 for part in command_paths)
    report["quality_gate_criteria"] = {
        "per_family_raw_recall": .80, "per_family_accepted_recall": .60,
        "global_and_per_family_accepted_precision": .95, "maximum_unknown_false_accept_rate": .01,
        "finite_command_recall": .60, "finite_command_precision": .95,
        "scope": "selected command paths only; VAD-only does not establish command quality"}
    report["quality_gate_note"] = "Software quality gates are separate from production readiness. Target-device, live microphone, reviewed-reference and dynamic-catalog acceptance remain unestablished."
    return report, predictions


def validate_dataset(config, root):
    root = Path(root)
    audit = audit_data(root)
    split, maximum = config["validation"]["split"], config["validation"]["max_examples"]
    rows = [r for r in read_rows(root / "data/vcm.jsonl") if r["split"] == split]
    if maximum:
        rows = rows[:maximum]
    refs = {r["id"]: r for r in read_rows(root / "data/references.jsonl")}
    report, predictions = evaluate(config, root / "artifacts", rows, root, refs)
    report.update(split=split, limited=bool(maximum), data_audit=audit)
    out = root / ".cache/validation/predictions.jsonl"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text("".join(json.dumps(r) + "\n" for r in predictions), encoding="utf-8")
    report["prediction_log"] = ".cache/validation/predictions.jsonl"
    return report


def validate_samples(config, bundle, sample_root):
    sample_root = Path(sample_root)
    rows = read_rows(sample_root / "samples.jsonl")
    refs = {r["id"]: r["annotation"] for r in rows}
    report, _ = evaluate(config, bundle, rows, sample_root, refs)
    report["scope"] = "packaged held-out smoke samples; not the full validation set"
    return report


### `tinyvcm.launcher`


In [ ]:
%%module tinyvcm.launcher
"""Self-contained pipeline: setup, component checks, microphone or audio-file inference."""
import argparse
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import subprocess
import sys
import tempfile
import venv
import zipfile


def materialize(archive_path):
    archive_path = Path(archive_path).resolve()
    fingerprint = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    destination = archive_path.parent / ".runtime" / fingerprint[:20]
    with zipfile.ZipFile(archive_path) as archive:
        names = archive.namelist()
        manifest = json.loads(archive.read("manifest.json"))
        if len(names) != len(set(names)) or set(names) != {*manifest["files"], "manifest.json"}:
            raise ValueError("Incomplete or duplicate pipeline archive entries")
        for name, expected in manifest["files"].items():
            parts = PurePosixPath(name).parts
            if (not parts or ".." in parts or "\\" in name or name.startswith("/")
                    or parts[0] not in {"__main__.py", "tinyvcm", "model", "samples"}):
                raise ValueError(f"Invalid pipeline archive entry: {name}")
            content = archive.read(name)
            if len(content) != expected["bytes"] or hashlib.sha256(content).hexdigest() != expected["sha256"]:
                raise ValueError(f"Pipeline checksum mismatch: {name}")
            if parts[0] not in {"model", "samples"}:
                continue
            path = destination / name
            if path.is_file() and path.read_bytes() == content:
                continue
            path.parent.mkdir(parents=True, exist_ok=True)
            with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as stream:
                temporary = Path(stream.name); stream.write(content)
            try:
                os.replace(temporary, path)
            finally:
                temporary.unlink(missing_ok=True)
    return destination, manifest


def setup(root, argv):
    parser = argparse.ArgumentParser(description="Install into a local .venv; bundled Pi wheels are used offline by default")
    parser.add_argument("--system-packages", action="store_true", help="install Pi OS Python/audio packages with apt")
    parser.add_argument("--online", action="store_true", help="fetch Python dependencies instead of using bundled wheels")
    args = parser.parse_args(argv)
    if sys.version_info[:2] not in {(3, 11), (3, 12), (3, 13)}:
        raise RuntimeError("Use CPython 3.11, 3.12 or 3.13")
    if args.system_packages:
        if not sys.platform.startswith("linux"):
            raise ValueError("--system-packages supports Raspberry Pi OS/Debian only")
        subprocess.run(["sudo", "apt-get", "update"], check=True)
        subprocess.run(["sudo", "apt-get", "install", "-y", "python3-venv", "libportaudio2", "libsndfile1"], check=True)
    environment = root / ".venv"
    python = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    if not python.exists():
        venv.EnvBuilder(with_pip=True).create(environment)
    command = [str(python), "-m", "pip", "install", "--disable-pip-version-check", "--only-binary=:all:", "--require-hashes", "-r", str(root / "requirements.lock")]
    if not args.online:
        if not (root / "wheels").is_dir():
            raise RuntimeError("No bundled wheels; republish with include_wheels=true or use setup --online")
        command += ["--no-index", "--find-links", str(root / "wheels")]
    subprocess.run(command, check=True)
    print("Installed. Run python3 pipeline.pyz --test, then --list-mics and --mic NUMBER.", flush=True)


def activate(root):
    environment = root / ".venv"
    python = environment / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    if python.exists() and Path(sys.prefix).resolve() != environment.resolve():
        command = [str(python), str(Path(sys.argv[0]).resolve()), *sys.argv[1:]]
        if os.name == "nt":
            raise SystemExit(subprocess.call(command))
        os.execv(str(python), command)


def emit(value):
    print(json.dumps(value, ensure_ascii=False, allow_nan=False), flush=True)


def main():
    archive = Path(sys.argv[0]).resolve()
    root = archive.parent
    if sys.argv[1:2] == ["setup"]:
        materialize(archive)
        setup(root, sys.argv[2:])
        return
    parser = argparse.ArgumentParser(description=__doc__)
    mode = parser.add_mutually_exclusive_group(required=True)
    mode.add_argument("--test", action="store_true", help="run configured checks and packaged human-speech samples")
    mode.add_argument("--list-mics", action="store_true")
    mode.add_argument("--mic", type=int, help="numeric microphone index")
    mode.add_argument("--wav", type=Path, help="decode/resample an audio file and run the configured stages")
    parser.add_argument("--config", type=Path, default=root / "pipeline.toml")
    parser.add_argument("--channel", type=int, default=0)
    parser.add_argument("--capture-rate", type=int)
    args = parser.parse_args()
    if args.channel < 0 or args.mic is not None and args.mic < 0 or args.capture_rate is not None and args.capture_rate <= 0:
        parser.error("Microphone/channel indices must be nonnegative and capture rate positive")
    activate(root)
    destination, manifest = materialize(archive)
    from tinyvcm.config import load_config
    config = load_config(args.config.resolve())
    if args.list_mics:
        import sounddevice as sd
        hosts = sd.query_hostapis()
        emit({"microphones": [{"index": i, "name": d["name"], "channels": d["max_input_channels"],
              "default_sample_rate": d["default_samplerate"], "host_api": hosts[d["hostapi"]]["name"]}
              for i, d in enumerate(sd.query_devices()) if d["max_input_channels"] > 0]})
        return
    if args.test:
        from tinyvcm.validation import run_checks, validate_samples
        result = {"components": manifest["components"], "selected_checks": config["tests"],
                  "tests": run_checks(config, destination / "model"),
                  "sample_validation": validate_samples(config, destination / "model", destination / "samples"),
                  "hardware": Path("/proc/device-tree/model").read_text(encoding="utf-8").strip("\x00")
                     if Path("/proc/device-tree/model").exists() else sys.platform,
                  "live_microphone_tested": False}
        (root / "launch-validation.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
        emit(result)
        if config["validation"]["require_quality_gate"] and not result["sample_validation"]["quality_gate_passed"]:
            raise RuntimeError("Quality gates unmet; see launch-validation.json")
        return
    from tinyvcm.pipeline import Pipeline
    pipeline = Pipeline(destination / "model", config)
    if args.wav:
        from tinyvcm.audio import load_audio
        pcm, rate = load_audio(args.wav.expanduser().resolve(), channel=args.channel)
        emit({"input_sample_rate": rate, "results": pipeline.process(pcm)})
        return
    from tinyvcm.capture import listen
    emit({"stage": "service", "status": "starting", "microphone": args.mic, "runtime": config["runtime"]})
    try:
        listen(pipeline, args)
    finally:
        emit({"stage": "service", "status": "stopped"})


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        pass
    except Exception as exc:
        print(f"Pipeline failed: {exc}\nInstall once with python3 pipeline.pyz setup. "
              "Pi OS prerequisites: python3-venv, libportaudio2, libsndfile1.", file=sys.stderr)
        sys.exit(1)


## Trainable model architecture and continuation implementation


### `tinyvcm.vcm`


In [ ]:
%%module tinyvcm.vcm
"""Small causal audio classifier. No ASR model, token decoder, or transcript input."""
from __future__ import annotations

import torch
from torch import nn
from torch.nn import functional as F
from einops import rearrange, reduce, pack


def valid_frames(x, lengths):
    """Validate eager inputs; exported callers must supply 1 <= length <= frames."""
    if not torch.jit.is_tracing():
        if (x.ndim != 3 or lengths.ndim != 1 or lengths.shape[0] != x.shape[0]
                or lengths.dtype not in {torch.int32, torch.int64}
                or torch.any(lengths < 1) or torch.any(lengths > x.shape[-1])):
            raise ValueError("Each utterance requires 1 <= valid length <= frame count")
    return rearrange(torch.arange(x.shape[-1], device=x.device), "t -> 1 1 t") < rearrange(lengths, "b -> b 1 1")


class FrameNorm(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        return rearrange(self.norm(rearrange(x, "b c t -> b t c")), "b t c -> b c t")


class CausalConv(nn.Module):
    def __init__(self, inputs, outputs, kernel, groups=1, dilation=1):
        super().__init__()
        self.left = (kernel - 1) * dilation
        self.inputs = inputs
        self.conv = nn.Conv1d(inputs, outputs, kernel, groups=groups, dilation=dilation, bias=False)

    def forward(self, x):
        return self.conv(F.pad(x, (self.left, 0)))

    def step(self, x, cache):
        joined, _ = pack([cache, x], "b c *")
        return self.conv(joined), joined[:, :, -self.left:]


class ResidualBlock(nn.Module):
    def __init__(self, width, kernel, dropout, dilation=1):
        super().__init__()
        self.depthwise = nn.ModuleList([CausalConv(width, width, kernel, groups=width, dilation=dilation) for _ in range(2)])
        self.pointwise = nn.ModuleList([nn.Conv1d(width, width, 1, bias=False) for _ in range(2)])
        self.norms = nn.ModuleList([FrameNorm(width) for _ in range(2)])
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        residual = x
        for depth, point, norm in zip(self.depthwise, self.pointwise, self.norms):
            x = self.dropout(F.silu(norm(point(depth(x)))))
        return (x + residual) * (2 ** -0.5)

    def step(self, x, caches):
        residual = x
        next_caches = []
        for depth, point, norm, cache in zip(self.depthwise, self.pointwise, self.norms, caches):
            x, next_cache = depth.step(x, cache)
            next_caches.append(next_cache)
            x = F.silu(norm(point(x)))
        return (x + residual) * (2 ** -0.5), next_caches


class VCM(nn.Module):
    def __init__(self, classes, width=128, kernels=(5, 7, 9, 11, 13), dropout=0.1, pooling="mean", dilations=None, keyword_classes=0, evidence_gate=None):
        super().__init__()
        self.classes, self.width = classes, width
        self.pooling = pooling
        if pooling not in {"mean", "mean_max", "attention"}:
            raise ValueError(f"Unsupported pooling: {pooling}")
        dilations = list(dilations or [1] * len(kernels))
        if len(dilations) != len(kernels) or any(d < 1 for d in dilations):
            raise ValueError("Provide one positive dilation per convolution block")
        self.stem = CausalConv(40, width, 5)
        self.stem_norm = FrameNorm(width)
        self.blocks = nn.ModuleList([ResidualBlock(width, k, dropout, d) for k, d in zip(kernels, dilations)])
        if pooling == "attention":
            self.attention = nn.Conv1d(width, 1, 1)
        self.head = nn.Linear(width * (2 if pooling == "mean_max" else 1), classes)
        self.config = {"classes": classes, "width": width, "kernels": list(kernels), "dropout": dropout, "pooling": pooling, "dilations": dilations}
        if keyword_classes:
            self.keyword_head = nn.Linear(self.head.in_features, keyword_classes)
            self.config["keyword_classes"] = keyword_classes
        self.evidence_gate = evidence_gate
        if evidence_gate:
            if len(evidence_gate["words"]) != keyword_classes or len(evidence_gate["intents"]) != classes:
                raise ValueError("Evidence vocabulary must match both trained heads")
            self.config["evidence_gate"] = evidence_gate

    def attention_weights(self, hidden):
        # Bounded positive weights keep full, prefix, and cached streaming sums identical.
        return self.attention(hidden).clamp(-10, 10).exp()

    def encode(self, x):
        x = F.silu(self.stem_norm(self.stem(x)))
        for block in self.blocks:
            x = block(x)
        return x

    def embedding(self, x, lengths):
        valid = valid_frames(x, lengths)
        hidden = self.encode(x)
        pooled = reduce(hidden * valid, "b c t -> b c", "sum") / rearrange(lengths, "b -> b 1")
        if self.pooling == "attention":
            weights = self.attention_weights(hidden) * valid
            pooled = reduce(hidden * weights, "b c t -> b c", "sum") / reduce(weights, "b c t -> b c", "sum").clamp_min(1e-8)
        if self.pooling == "mean_max":
            maximum = reduce(hidden.masked_fill(~valid, -1e4), "b c t -> b c", "max")
            pooled, _ = pack([pooled, maximum], "b *")
        return pooled

    def forward(self, x, lengths):
        return self.classify_embedding(self.embedding(x, lengths))

    def classify_embedding(self, pooled):
        logits = self.head(pooled)
        if not self.evidence_gate:
            return logits
        word_probabilities = self.keyword_head(pooled).sigmoid()
        words = self.evidence_gate["words"]
        def any_word(names):
            indices = [words.index(w) for w in names.split() if w in words]
            return reduce(word_probabilities[..., indices], "... k -> ...", "max") if indices else torch.zeros_like(word_probabilities[..., 0])
        light = any_word("light lights lamp lamps brightness")
        volume_absent = 1 - any_word("volume")
        up = torch.maximum(any_word("brighten brighter"), any_word("up increase raise more higher") * light)
        down = torch.maximum(any_word("dim dimmer dimming darker darken"), any_word("down decrease lower less reduce") * torch.maximum(light, any_word("bright brighter")))
        up = up * (1 - any_word("less down decrease lower dim")) * volume_absent
        down = down * volume_absent
        color = any_word("color colors colour colours red orange yellow green blue purple pink cyan white warm cool")
        color = color * (1 - any_word("play music song songs what whether confirm"))
        evidence = {"BRIGHTNESS_UP": up, "BRIGHTNESS_DOWN": down, "COLOR_SET": color}
        adjustments = []
        for label in self.evidence_gate["intents"]:
            if label in evidence:
                p = evidence[label].clamp(.01, .99)
                adjustments.append(p.log() - (1-p).log())
            else:
                adjustments.append(torch.zeros_like(word_probabilities[..., 0]))
        return logits + self.evidence_gate["strength"] * rearrange(adjustments, "c ... -> ... c")

    def forward_with_keywords(self, x, lengths):
        """Training-only word supervision. The runtime still takes audio alone."""
        pooled = self.embedding(x, lengths)
        return self.classify_embedding(pooled), self.keyword_head(pooled)

    def prefix_logits(self, x):
        hidden = self.encode(x)
        pooled = hidden.cumsum(-1) / rearrange(torch.arange(1, x.shape[-1] + 1, device=x.device), "t -> 1 1 t")
        if self.pooling == "attention":
            weights = self.attention_weights(hidden)
            pooled = (hidden * weights).cumsum(-1) / weights.cumsum(-1).clamp_min(1e-8)
        if self.pooling == "mean_max":
            pooled, _ = pack([pooled, hidden.cummax(-1).values], "b * t")
        return self.classify_embedding(rearrange(pooled, "b c t -> b t c"))

    def initial_state(self, device="cpu"):
        states = [torch.zeros(1, self.width, device=device), torch.zeros(1, 1, device=device)]
        if self.pooling == "mean_max":
            states.append(torch.full((1, self.width), -1e4, device=device))
        states.append(torch.zeros(1, 40, self.stem.left, device=device))
        for block in self.blocks:
            states.extend(torch.zeros(1, self.width, d.left, device=device) for d in block.depthwise)
        return states

    def step(self, x, states):
        total, count = states[:2]
        offset = 3 if self.pooling == "mean_max" else 2
        x, stem_cache = self.stem.step(x, states[offset])
        x = F.silu(self.stem_norm(x))
        new_caches = [stem_cache]
        cursor = offset + 1
        for block in self.blocks:
            x, caches = block.step(x, states[cursor:cursor + 2])
            cursor += 2
            new_caches.extend(caches)
        if self.pooling == "attention":
            weights = self.attention_weights(x)
            total = total + reduce(x * weights, "b c t -> b c", "sum")
            count = count + reduce(weights, "b c t -> b c", "sum")
        else:
            total = total + reduce(x, "b c t -> b c", "sum")
            count = count + x.shape[-1]
        pooled = total / count
        next_states = [total, count]
        if self.pooling == "mean_max":
            maximum = torch.maximum(states[2], reduce(x, "b c t -> b c", "max"))
            pooled, _ = pack([pooled, maximum], "b *")
            next_states.append(maximum)
        return self.classify_embedding(pooled), [*next_states, *new_caches]


class StreamingExport(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, features, *states):
        logits, states = self.model.step(features, states)
        return (logits, *states)


class EnsembleVCM(nn.Module):
    """Average independent acoustic classifiers with the same input normalization."""
    def __init__(self, ensemble):
        super().__init__()
        self.members = nn.ModuleList([VCM(**config) for config in ensemble])
        if len(self.members) < 2 or len({m.classes for m in self.members}) != 1:
            raise ValueError("An ensemble requires at least two classifiers with the same vocabulary size")
        self.config = {"ensemble": ensemble}
        self.state_counts = [len(m.initial_state()) for m in self.members]

    def forward(self, x, lengths):
        logits, _ = pack([m(x, lengths) for m in self.members], "b c *")
        return reduce(logits, "b c n -> b c", "mean")

    def prefix_logits(self, x):
        logits, _ = pack([m.prefix_logits(x) for m in self.members], "b t c *")
        return reduce(logits, "b t c n -> b t c", "mean")

    def initial_state(self, device="cpu"):
        return [state for member in self.members for state in member.initial_state(device)]

    def step(self, x, states):
        outputs, updated = [], []
        cursor = 0
        for member, count in zip(self.members, self.state_counts):
            logits, next_states = member.step(x, states[cursor:cursor + count])
            outputs.append(logits)
            updated.extend(next_states)
            cursor += count
        logits, _ = pack(outputs, "b c *")
        return reduce(logits, "b c n -> b c", "mean"), updated


def build_vcm(config):
    return EnsembleVCM(**config) if "ensemble" in config else VCM(**config)


### `tinyvcm.slot_model`


In [ ]:
%%module tinyvcm.slot_model
"""One acoustic backbone pass for the original intents and finite parameter heads."""
from __future__ import annotations

import torch
from torch import nn
from torch.nn import functional as F
from einops import pack, reduce

from .vcm import build_vcm


def masked_head_losses(outputs, targets, masks):
    """Normalize each task by its own labels, including a differentiable empty loss."""
    losses = {}
    for name, logits in outputs.items():
        mask = masks[name]
        if mask.any():
            losses[name] = F.cross_entropy(logits[mask], targets[name][mask])
        else:
            losses[name] = reduce(logits, "b c ->", "sum") * 0.
    return losses


class SlotHeads(nn.Module):
    def __init__(self, features, sizes, classes, hidden=192, adapt_intent=False):
        super().__init__()
        self.names = list(sizes)
        self.project = nn.Sequential(nn.LayerNorm(features), nn.Linear(features, hidden), nn.SiLU(), nn.Dropout(.15))
        self.slots = nn.ModuleDict({name: nn.Linear(hidden, size) for name, size in sizes.items()})
        self.adapt_intent = adapt_intent
        self.intent = nn.Sequential(nn.LayerNorm(features), nn.Linear(features, hidden), nn.SiLU(), nn.Dropout(.15), nn.Linear(hidden, classes))
        nn.init.zeros_(self.intent[-1].weight)
        nn.init.zeros_(self.intent[-1].bias)
        if not adapt_intent:
            for parameter in self.intent.parameters():
                parameter.requires_grad_(False)

    def forward(self, features, original_logits):
        hidden = self.project(features)
        intent = original_logits + self.intent(features) if self.adapt_intent else original_logits
        return (intent, *[self.slots[name](hidden) for name in self.names])


class SlotVCM(nn.Module):
    def __init__(self, backbone_config, vocabulary, hidden=192, adapt_intent=False):
        super().__init__()
        self.backbone = build_vcm(backbone_config)
        members = list(self.backbone.members) if hasattr(self.backbone, "members") else [self.backbone]
        width = sum(member.head.in_features for member in members)
        self.heads = SlotHeads(width, {k: len(v) for k, v in vocabulary.items()}, members[0].classes, hidden, adapt_intent)
        self.config = {"backbone_config": backbone_config, "vocabulary": vocabulary, "hidden": hidden, "adapt_intent": adapt_intent}

    def encode(self, x, lengths):
        members = list(self.backbone.members) if hasattr(self.backbone, "members") else [self.backbone]
        embeddings = [member.embedding(x, lengths) for member in members]
        logits = [member.classify_embedding(value) for member, value in zip(members, embeddings)]
        joined, _ = pack(embeddings, "b *")
        combined, _ = pack(logits, "b c *")
        return joined, reduce(combined, "b c m -> b c", "mean")

    def forward(self, x, lengths):
        return self.heads(*self.encode(x, lengths))


### `tinyvcm.data`


In [ ]:
%%module tinyvcm.data
from pathlib import Path
import os
import json
from .text import normalize_text
ROOT = Path(os.environ.get("SMART_SPEAKER_ROOT", str(Path.cwd()))).resolve()

def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


### `tinyvcm.train_helpers`


In [ ]:
%%module tinyvcm.train_helpers
from concurrent.futures import ThreadPoolExecutor
import math
import random
import time
import numpy as np
import torch
from einops import rearrange, reduce
from .audio import DEFAULT, cached_features
from .data import ROOT
_FEATURE_MEMORY = {}
def training_features(record):
    value = _FEATURE_MEMORY.get(record["id"])
    return value if value is not None else cached_features(record, ROOT)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(4)
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


def warm_cache(rows, workers=6, memory_mib=0):
    started = last = time.monotonic()
    memory_bytes = sum(array.nbytes for array in _FEATURE_MEMORY.values()) if memory_mib else 0
    def cache(record):
        array = training_features(record)
        return array if memory_mib else array.shape[1]
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for i, value in enumerate(pool.map(cache, rows), 1):
            frames = value.shape[1] if memory_mib else value
            rows[i - 1]["frames"] = frames
            if memory_mib and rows[i - 1]["id"] not in _FEATURE_MEMORY:
                memory_bytes += value.nbytes
                if memory_bytes > memory_mib * 2**20:
                    raise RuntimeError(f"Feature cache exceeds the requested {memory_mib} MiB budget")
                value.flags.writeable = False
                _FEATURE_MEMORY[rows[i - 1]["id"]] = value
            if time.monotonic() - last > 20:
                print(f"FEATURES {i}/{len(rows)} elapsed={time.monotonic()-started:.0f}s", flush=True)
                last = time.monotonic()
    if memory_mib:
        print(f"RAM FEATURE CACHE {len(_FEATURE_MEMORY)} arrays, {memory_bytes/2**20:.1f} MiB", flush=True)


def normalization(rows):
    if not rows or any(row.get("split") != "train" for row in rows):
        raise ValueError("Normalization fitting requires nonempty training-only records")
    total = np.zeros(40, dtype=np.float64)
    squares = np.zeros(40, dtype=np.float64)
    count = 0
    for row in rows:
        x = training_features(row).astype(np.float64)
        total += reduce(x, "m t -> m", "sum")
        squares += reduce(x * x, "m t -> m", "sum")
        count += x.shape[1]
    mean = total / count
    std = np.sqrt(np.maximum(squares / count - mean * mean, 1e-4))
    return {"mean": mean.tolist(), "std": std.tolist(), "training_frames": count}


def batches(rows, batch_size, rng=None, frame_budget=16000, indices=None):
    indices = list(range(len(rows))) if indices is None else list(indices)
    if rng is not None:
        rng.shuffle(indices)
    output = []
    for offset in range(0, len(indices), batch_size * 32):
        bucket = sorted(indices[offset:offset + batch_size * 32], key=lambda i: rows[i].get("frames", rows[i]["duration"] * 100))
        current, maximum = [], 0
        for index in bucket:
            length = rows[index].get("frames", math.ceil(rows[index]["duration"] * 100))
            if current and (len(current) >= batch_size or max(maximum, length) * (len(current) + 1) > frame_budget):
                output.append(current)
                current, maximum = [], 0
            current.append(index)
            maximum = max(maximum, length)
        if current:
            output.append(current)
    if rng is not None:
        rng.shuffle(output)
    return output


def make_batch(rows, indices, stats, augment=False, rng=None, speed_augment=False, frequency_warp=False, gain_augment_db=0.):
    arrays = [training_features(rows[i]) for i in indices]
    if augment and speed_augment:
        from scipy.ndimage import zoom
        # Feature-domain temporal resampling is explicitly distinct from waveform augmentation.
        arrays = [zoom(x, (1, rng.uniform(.9, 1.1)), order=1, prefilter=False) for x in arrays]
    if augment and frequency_warp:
        from scipy.ndimage import affine_transform
        warped = []
        for x in arrays:
            scale = rng.uniform(.9, 1.1)
            warped.append(affine_transform(x, [[scale, 0], [0, 1]], offset=[(1-scale)*19.5, 0],
                                           order=1, mode="nearest", prefilter=False))
        arrays = warped
    lengths = torch.tensor([x.shape[1] for x in arrays], dtype=torch.long)
    maximum = int(math.ceil(int(lengths.max()) / 32) * 32)
    batch = np.zeros((len(arrays), 40, maximum), dtype=np.float32)
    mean = rearrange(np.array(stats["mean"], dtype=np.float32), "m -> m 1")
    std = rearrange(np.array(stats["std"], dtype=np.float32), "m -> m 1")
    for j, x in enumerate(arrays):
        if augment and gain_augment_db:
            # Power-log features: 10 dB is ln(10), with the same frontend floor.
            # Training-only augmentation; the deployment frontend is unchanged.
            x = np.maximum(x + rng.uniform(-gain_augment_db, gain_augment_db) * np.log(10) / 10,
                           np.log(DEFAULT.log_floor))
        x = (x - mean) / std
        if augment:
            x = x.copy()
            # Feature-space augmentation is explicitly reported; no synthetic room claim.
            x += rng.uniform(-0.15, 0.15)
            width = rng.randrange(0, 6)
            begin = rng.randrange(0, 40 - width + 1)
            x[begin:begin + width, :] = 0
            width = rng.randrange(0, min(25, x.shape[1] // 10) + 1)
            begin = rng.randrange(0, x.shape[1] - width + 1)
            x[:, begin:begin + width] = 0
        batch[j, :, :x.shape[1]] = x
    return torch.from_numpy(batch), lengths

from .text import normalize_text
from .validation import edit_distance



### `tinyvcm.finite_policy`


In [ ]:
%%module tinyvcm.finite_policy
from collections import Counter
import torch
from .family_contract import LABELS, family_candidate, route_family
from .slot_contract import apply_value_policy
from .routing import PARAMETER_COMMANDS, parse_parameters

def finite_reference(annotation: dict) -> dict | None:
    reference = annotation["reference"]
    if not reference or reference["status"] != "accepted":
        return None
    if set(reference["command"]["arguments"]) & {"title", "artist", "selection_text"}:
        return None
    return reference["command"]

def threshold(scores: list[float], correct: list[bool], precision: float = .95) -> float:
    options = []
    for value in sorted(set(scores)):
        accepted = [good for score, good in zip(scores, correct) if score >= value]
        if len(accepted) >= 20 and sum(accepted)/len(accepted) >= precision:
            options.append((sum(accepted), -len(accepted), value))
    return max(options)[2] if options else 1.1

def slot_records(outputs: tuple, vocab: dict) -> list[dict]:
    values = {name: torch.softmax(output, -1).cpu().numpy()
              for name, output in zip(vocab, outputs[1:])}
    return [{name: p[i] for name, p in values.items()} for i in range(len(outputs[0]))]

def provisional_slots(rows, annotations, accepted, slots, schema, catalog):
    policy = {"value_thresholds": {}, "acceptance_thresholds": {},
              "custom_media_threshold": 1.1, "custom_guard_threshold": 1.1,
              "clarification_thresholds": {}, "partition": "development policy_fit only"}
    candidates = [family_candidate(r["predicted"], p, schema, catalog)
                  if r["predicted"] != "UNKNOWN" else None for r, p in zip(accepted, slots)]
    for name, values in schema["vocabulary"].items():
        if name == "guard":
            continue
        policy["value_thresholds"][name] = {}
        for value in values:
            scores, correct = [], []
            for row, annotation, intent, candidate in zip(rows, annotations, accepted, candidates):
                if (not intent["accepted"] or intent["predicted"] != row["command"] or not candidate
                        or candidate["values"].get(name) != value):
                    continue
                gold = annotation["labels"].get(name)
                if gold is not None:
                    scores.append(candidate["confidence"][name])
                    correct.append(gold == value)
            policy["value_thresholds"][name][value] = threshold(scores, correct, .99 if name == "color" else .95)
    for family in LABELS[1:]:
        scores, correct = [], []
        for row, annotation, intent, candidate in zip(rows, annotations, accepted, candidates):
            if not intent["accepted"] or intent["predicted"] != family or not candidate:
                continue
            candidate = apply_value_policy(candidate, policy)
            if candidate["score"] > 0:
                scores.append(candidate["score"])
                correct.append(row["command"] == family and candidate["result"]["command"] == finite_reference(annotation))
        policy["acceptance_thresholds"][family] = threshold(scores, correct)
    # Custom routing remains off; this pilot does not establish a dynamic-catalog gate.
    return policy

def command_metrics(rows, annotations, accepted, slots, config, catalog):
    emitted = correct = resolvable = 0
    per_family = {c: Counter() for c in LABELS}
    errors = []
    for row, annotation, intent, values in zip(rows, annotations, accepted, slots):
        reference = finite_reference(annotation)
        resolvable += reference is not None
        per_family[row["command"]]["resolvable"] += reference is not None
        decision = {"decision": "rejected"}
        if intent["accepted"]:
            decision = route_family(intent["predicted"], values, config, catalog, "fp32")
            if decision["decision"] == "fallback" and intent["predicted"] not in PARAMETER_COMMANDS:
                decision = {"decision": "accepted", **parse_parameters(intent["predicted"], "", catalog)}
        command = decision.get("command") if decision["decision"] == "accepted" else None
        success = command is not None and row["command"] == intent["predicted"] and command == reference
        emitted += command is not None
        correct += success
        per_family[row["command"]]["correct_complete"] += success
        per_family[row["command"]]["emitted"] += command is not None
        errors.append({"id": row["id"], "gold": row["command"], "predicted": intent["predicted"],
                       "intent_accepted": intent["accepted"], "command": command,
                       "reference": reference, "correct_complete": success,
                       "decision": decision["decision"]})
    return {"correct_complete": correct, "resolvable_finite_references": resolvable,
            "emitted": emitted, "exact_command_success": correct/max(1, resolvable),
            "accepted_command_precision": correct/max(1, emitted), "per_family": per_family,
            "command_promotion_assessed": False,
            "denominator": "all silver resolvable finite commands; open-text media reported as clarification"}, errors


### `tinyvcm.training`


In [ ]:
%%module tinyvcm.training
"""Continue the current checkpoint; train/dev only, with explicit artifact replacement."""
import copy
import hashlib
import json
from pathlib import Path
import random
import shutil
import tempfile
import time
import numpy as np
import torch
from einops import pack
from .data import read_jsonl
from .train_helpers import batches, make_batch, seed_everything, warm_cache


def subset(rows, split, maximum):
    selected = [r for r in rows if r["split"] == split]
    return selected[:maximum] if maximum else selected


def fit_vcm(config, root, output):
    from .slot_model import SlotVCM, masked_head_losses
    from .finite_policy import provisional_slots, slot_records, command_metrics
    from .metrics import policy_rows
    from .slot_contract import catalog_signature
    cfg = config["training"]
    ck = torch.load(root / "artifacts/vcm.pt", map_location="cpu", weights_only=False)
    metadata = json.loads((root / "artifacts/bundle.json").read_text(encoding="utf-8"))
    schema = json.loads((root / "data/contract.json").read_text(encoding="utf-8"))
    refs = {r["id"]: r for r in read_jsonl(root / "data/references.jsonl")}
    rows = read_jsonl(root / "data/vcm.jsonl")
    training = subset(rows, "train", cfg["max_train_examples"])
    development = subset(rows, "dev", cfg["max_dev_examples"])
    folds = json.loads((root / "data/development-folds.json").read_text(encoding="utf-8"))
    # If a debug limit is used, still take examples from both fixed disjoint folds.
    if cfg["max_dev_examples"]:
        maximum = max(1, cfg["max_dev_examples"] // 2)
        development = [r for fold in ("policy_fit", "selection")
                       for r in [x for x in rows if x["split"] == "dev" and folds[x["id"]] == fold][:maximum]]
    rows = training + development
    if not training or not development:
        raise ValueError("Training requires nonempty train and development records")
    train_ids = list(range(len(training)))
    fit_ids = [i for i, r in enumerate(rows) if r["split"] == "dev" and folds[r["id"]] == "policy_fit"]
    select_ids = [i for i, r in enumerate(rows) if r["split"] == "dev" and folds[r["id"]] == "selection"]
    if not fit_ids or not select_ids:
        raise ValueError("Need both grouped development folds")
    device = cfg["device"]
    model = SlotVCM(**ck["config"]).to(device).eval()
    model.load_state_dict(ck["model"])
    for parameter in model.backbone.parameters():
        parameter.requires_grad_(False)
    warm_cache(rows)
    vectors = torch.empty(len(rows), model.heads.project[0].normalized_shape[0])
    logits = torch.empty(len(rows), len(ck["labels"]))
    with torch.inference_mode():
        for n, ids in enumerate(batches(rows, cfg["batch_size"])):
            x, lengths = make_batch(rows, ids, ck["normalization"])
            a, b = model.encode(x.to(device), lengths.to(device))
            vectors[ids], logits[ids] = a.cpu(), b.cpu()
            if n % 100 == 0:
                print(f"VCM ENCODE batch={n}", flush=True)
    vectors, logits = vectors.to(device), logits.to(device)
    vocab = ck["config"]["vocabulary"]
    annotations = [refs[r["id"]] for r in rows]
    targets = {name: torch.tensor([values.index(a["labels"][name]) if a["labels"][name] is not None else 0
                                  for a in annotations], device=device) for name, values in vocab.items()}
    masks = {name: torch.tensor([a["labels"][name] is not None for a in annotations], device=device) for name in vocab}
    # Frozen family encoder => preserve the exact current family acceptance policy.
    intent_policy = ck["intent_policy"]
    def accepted(ids):
        return policy_rows([{"id": rows[i]["id"], "gold": rows[i]["command"],
                             "probabilities": torch.softmax(logits[i], -1).cpu().tolist()} for i in ids],
                           ck["labels"], intent_policy)
    accepted_fit, accepted_selection = accepted(fit_ids), accepted(select_ids)
    optimizer = torch.optim.AdamW([p for p in model.heads.parameters() if p.requires_grad],
                                 lr=cfg["learning_rate"], weight_decay=.01)
    best, best_state, history, stale = None, None, [], 0
    groups = {label: [i for i in train_ids if rows[i]["command"] == label] for label in ck["labels"]}
    groups = {label: ids for label, ids in groups.items() if ids}
    for epoch in range(1, cfg["vcm_epochs"] + 1):
        model.heads.train()
        rng = random.Random(cfg["seed"] * 1000 + epoch)
        draws = [rng.choice(ids) for ids in groups.values() for _ in range(min(600, len(training)))]
        rng.shuffle(draws)
        losses_seen = []
        for offset in range(0, len(draws), cfg["batch_size"]):
            ids = draws[offset:offset + cfg["batch_size"]]
            outputs = model.heads(vectors[ids], logits[ids])
            losses = masked_head_losses(dict(zip(vocab, outputs[1:])),
                {name: v[ids] for name, v in targets.items()}, {name: v[ids] for name, v in masks.items()})
            loss = sum(.5 * v for v in losses.values())
            optimizer.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.heads.parameters(), 3.)
            optimizer.step(); losses_seen.append(float(loss.detach()))
        model.heads.eval()
        with torch.inference_mode():
            fit_outputs = model.heads(vectors[fit_ids], logits[fit_ids])
            selection_outputs = model.heads(vectors[select_ids], logits[select_ids])
        torch.testing.assert_close(selection_outputs[0], logits[select_ids], atol=0., rtol=0.)
        fit_slots, selection_slots = slot_records(fit_outputs, vocab), slot_records(selection_outputs, vocab)
        policy = provisional_slots([rows[i] for i in fit_ids], [annotations[i] for i in fit_ids],
                                   accepted_fit, fit_slots, schema, metadata["catalog"])
        slot_config = {**schema, "catalog_signature": catalog_signature(metadata["catalog"]), "variants": {"fp32": policy}}
        command, _ = command_metrics([rows[i] for i in select_ids], [annotations[i] for i in select_ids],
            accepted_selection, selection_slots, slot_config, metadata["catalog"])
        rank = (command["accepted_command_precision"] >= .95, command["exact_command_success"],
                command["accepted_command_precision"])
        entry = {"epoch": epoch, "loss": sum(losses_seen)/max(1, len(losses_seen)), "commands": command}
        history.append(entry)
        if best is None or rank > best:
            best, stale = rank, 0
            best_state = {**ck, "model": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                          "epoch": epoch, "seed": cfg["seed"], "slot_policy": policy, "metrics": entry}
        else:
            stale += 1
        print(f"VCM epoch={epoch} exact={command['exact_command_success']:.4f}", flush=True)
        if stale >= cfg["patience"]:
            break
    torch.save(best_state, output / "vcm.pt")
    (output / "training-history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    metadata["acoustic_slots"]["variants"]["fp32"] = best_state["slot_policy"]
    return metadata, {"epochs": len(history), "selected_epoch": best_state["epoch"],
                      "train_examples": len(training), "dev_examples": len(development), "encoder_frozen": True}




def export_graphs(output, selected, metadata):
    """Export only the VCM; reject obsolete task names before opening files."""
    import onnx
    import onnxruntime as ort
    from .slot_model import SlotVCM
    if selected != ["vcm"]:
        raise ValueError("Only the VCM export is supported")
    ck = torch.load(output / "vcm.pt", map_location="cpu", weights_only=False)
    model = SlotVCM(**ck["config"]).eval()
    model.load_state_dict(ck["model"])
    x, lengths = torch.zeros(1, 40, 160), torch.tensor([160])
    path = output / "vcm.fp32.onnx"
    names = ["logits", *["slot_" + n for n in ck["config"]["vocabulary"]]]
    axes = {"features": {0: "batch", 2: "frames"}, "lengths": {0: "batch"},
            **{name: {0: "batch"} for name in names}}
    torch.onnx.export(model, (x, lengths), str(path),
                      input_names=["features", "lengths"], output_names=names,
                      dynamic_axes=axes, opset_version=17, dynamo=False)
    graph = onnx.load(str(path))
    for key, value in (("vcm_labels", ck["labels"]),
                       ("slot_vocabulary", ck["config"]["vocabulary"])):
        entry = graph.metadata_props.add()
        entry.key, entry.value = key, json.dumps(value)
    onnx.checker.check_model(graph)
    onnx.save(graph, str(path))
    options = ort.SessionOptions()
    options.intra_op_num_threads = 1
    session = ort.InferenceSession(str(path), sess_options=options,
                                   providers=["CPUExecutionProvider"])
    maximum = 0.0
    for frames in (61, 160, 317):
        x, lengths = torch.randn(1, 40, frames), torch.tensor([frames])
        with torch.inference_mode():
            expected = model(x, lengths)
        actual = session.run(None, {"features": x.numpy(), "lengths": lengths.numpy()})
        if len(expected) != len(actual):
            raise ValueError("VCM output count changed at export")
        for left, right in zip(expected, actual):
            np.testing.assert_allclose(left.numpy(), right, atol=2e-4, rtol=2e-4)
            maximum = max(maximum, float(np.max(np.abs(left.numpy() - right))))
    metadata["files"][path.name] = {
        "bytes": path.stat().st_size,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    metadata["vcm"].update(checkpoint="vcm.pt", checkpoint_sha256=
        hashlib.sha256((output / "vcm.pt").read_bytes()).hexdigest())
    return {"vcm": maximum}


def fit(config, root):
    root = Path(root); cfg = config["training"]
    if cfg["device"] == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA unavailable; set training.device = 'cpu' explicitly")
    selected = [task for task in ("vcm",) if cfg[task]]
    if not selected:
        raise ValueError("No training component selected")
    if cfg["update_artifacts"] and (cfg["max_train_examples"] or cfg["max_dev_examples"]):
        raise ValueError("Limited debug training cannot replace the current artifacts; set update_artifacts = false")
    seed_everything(cfg["seed"])
    parent = root / ".cache/training"; parent.mkdir(parents=True, exist_ok=True)
    output = Path(tempfile.mkdtemp(prefix="candidate-", dir=parent))
    for path in (root / "artifacts").iterdir():
        if path.is_file() and not path.name.startswith("asr"):
            shutil.copy2(path, output / path.name)
    metadata = json.loads((output / "bundle.json").read_text(encoding="utf-8"))
    result = {"selected": selected, "output": str(output.relative_to(root)),
              "continuation_from_current_weights": True, "calibration_or_test_used_for_fitting": False,
              "scope": "frozen-family encoder with finite-head continuation"}
    if "vcm" in selected:
        metadata, result["vcm"] = fit_vcm(config, root, output)
    result["export_max_errors"] = export_graphs(output, selected, metadata)
    metadata["experiment"].update(status="experimental_continuation", training=result)
    (output / "bundle.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    from .validation import run_checks
    test_config = copy.deepcopy(config); test_config["tests"] = {key: True for key in config["tests"]}
    result["checks"] = run_checks(test_config, output)
    if cfg["update_artifacts"]:
        # Keep a single previous-current recovery copy until the next successful fit.
        backup = parent / "previous-current"
        backup.mkdir(exist_ok=True)
        for path in (root / "artifacts").iterdir():
            if path.is_file() and not path.name.startswith("asr"):
                shutil.copy2(path, backup / path.name)
        for path in output.iterdir():
            if path.is_file() and path.name != "manifest.json":
                shutil.copy2(path, root / "artifacts" / path.name)
    result["artifacts_updated"] = cfg["update_artifacts"]
    return result


## Verify the current artifacts

Running these cells checks the exact saved artifacts and the selected real runtime components. Full held-out metrics are written above by `train.py` and to `validation.json`. Fixture checks exercise ASR routing without changing production thresholds. They are labelled separately from actual acoustic inference.


In [ ]:
from tinyvcm.config import load_config
from tinyvcm.validation import run_checks
manifest = train.verify_artifacts(PROJECT)
config = load_config(PROJECT / "pipeline.toml")
checks = run_checks(config, PROJECT / "artifacts")
print(manifest["name"])
print({name: result["status"] for name, result in checks.items()})


In [ ]:
import json
metadata = json.loads((PROJECT / "artifacts/bundle.json").read_text(encoding="utf-8"))
print("Models:", metadata["runtime_components"])
print("VCM:", metadata["vcm"]["labels"])
print("ASR:", metadata["asr"]["parameters"], "parameters")
print("Custom media threshold:", metadata["acoustic_slots"]["variants"]["fp32"]["custom_media_threshold"])
# Explicit training uses the same source: python train.py --fit


## Acceptance limits

No current human-speech or physical-Pi qualification is claimed. The historical low accepted-recall results must not be hidden by removal. Custom names require the separate catalog research path, not unrestricted transcription.
